
# Paper 2 — Goal 3 controlled Q+A measurement preflight

## Purpose

This notebook validates the **measurement side** of the controlled Goal 3 experiment while Goal 2 final completion is still running.

It is deliberately prohibited from loading diagnosis, bulbar scores, Goal 2 predictions, losses, or clinical model objects.

The preflight has two hard objectives:

1. **Baseline A reproduction:** an unmodified source re-segmented through the Goal 3 end-to-end path must reproduce the already frozen Bamboo acoustic representation A generated by the v2.1 extractor.
2. **Controlled Q+A execution:** a tiny outcome-blind pilot must successfully traverse sealed perturbation → re-segmentation → Q extraction → A extraction → ΔQ/ΔA with restart-safe outputs.

This notebook does **not** perform clinical inference and does **not** run the 199 diagnosis / 145 severity controlled experiment.

### Frozen upstream inputs

- Paper 1 code commit: `cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8`
- Acoustic extraction engine: `bamboo-acoustic-v2.1.0`
- A freeze: `a-bamboo-v2.1.0`
- Goal 3 Stage-B sealed perturbation manifest
- Outcome-blind 24-source Stage-A dose pilot
- Fixed five-fold Goal 3 cross-fit
- Fixed outer-training QCHAN reference construction

### Preflight sample

The first **two** deterministic outcome-blind pilot sources are used. For each source, the notebook evaluates:

- unmodified baseline;
- low and high dose for each of the six sealed perturbation structures;
- exemplar 1 only for stochastic/exemplar transforms.

This is a software/measurement preflight. It does not reselect doses and does not change the Stage-B seal.


**v1.0.1 correction:** the frozen v2.1 A extractor first merged overlapping/touching `primary_speech` rows before acoustic measurement. Baseline reproduction therefore compares fresh re-segmentation against that merged frozen representation, not against the raw segmentation-table row count.

In [1]:

from __future__ import annotations

import hashlib
import inspect
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import linalg, signal, stats

ENGINE_VERSION = "goal3-controlled-QA-preflight-v1.0.2"
PAPER1_COMMIT = "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"
ACOUSTIC_ENGINE = "bamboo-acoustic-v2.1.0"
A_FREEZE_VERSION = "a-bamboo-v2.1.0"
BASE_SEED = 20260825

EXPECTED = {
    "retained_recordings": 519,
    "participants": 224,
    "pilot_sources": 24,
    "outer_folds": 5,
    "sealed_transforms": 6,
    "sealed_dose_rows": 18,
    "preflight_sources": 2,
}

QDIST_GEOMETRY = "symmetric"

PROHIBITED_CLINICAL_COLUMNS = {
    "diagnosis",
    "Diagnosis",
    "y",
    "y_dx",
    "bulbar_score",
    "ALSFRS",
    "assessment_date",
    "age_at_recording_years",
    "sex",
    "prediction",
    "predicted",
    "loss",
    "absolute_error",
    "brier",
}

def find_project_root() -> Path:
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "data" / "processed" / "recording_table_phase0.csv").exists():
            return root
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {root}")

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "processed" / "recording_table_phase0.csv").exists()
            and (candidate / "outputs" / "goal3").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate Paper 2 repository root. "
        "Run from the Paper_2_Leakage/Code repository."
    )

ROOT = find_project_root()

PROCESSED = ROOT / "data" / "processed"
MANIFESTS = ROOT / "data" / "manifests"
INTERIM = ROOT / "data" / "interim"
RAW = ROOT / "data" / "raw"
EXTERNAL = ROOT / "external" / "quality_framework_features"

OUT = ROOT / "outputs" / "goal3" / "stageC_QA_preflight_v1_0"
TABLES = OUT / "tables"
AUDIT = OUT / "audit"
CHECKPOINTS = OUT / "checkpoints"
CACHE = OUT / "cache"

for directory in [OUT, TABLES, AUDIT, CHECKPOINTS, CACHE]:
    directory.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def stable_hash(*parts):
    payload = "|".join(str(part) for part in parts)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def stable_seed(*parts):
    return (
        BASE_SEED
        + int(stable_hash(*parts)[:8], 16)
    ) % (2**32 - 1)

def safe_slug(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")

def json_safe(value):
    if value is pd.NA:
        return None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.isoformat()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        return [json_safe(v) for v in value]
    return value

def atomic_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.tmp")
    temp.write_text(
        json.dumps(json_safe(payload), indent=2, sort_keys=True),
        encoding="utf-8",
    )
    os.replace(temp, path)

def atomic_csv(frame, path, allow_empty=False):
    path = Path(path)
    if frame.empty and not allow_empty:
        raise ValueError(f"Refusing to write empty table: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.name}.tmp")
    frame.to_csv(temp, index=False)
    if temp.stat().st_size == 0:
        raise IOError(f"Temporary CSV is empty: {temp}")
    os.replace(temp, path)

def read_csv_checked(path, required=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size == 0:
        raise RuntimeError(f"Empty CSV: {path}")
    frame = pd.read_csv(path, low_memory=False)
    if required:
        missing = set(required) - set(frame.columns)
        if missing:
            raise KeyError(f"{path.name}: missing columns {sorted(missing)}")
    return frame

def assert_no_clinical_columns(name, frame):
    leaked = PROHIBITED_CLINICAL_COLUMNS & set(frame.columns)
    if leaked:
        raise RuntimeError(
            f"{name} contains prohibited clinical/model columns: {sorted(leaked)}"
        )

print("Project root:", ROOT)
print("Engine:", ENGINE_VERSION)
print("Python:", sys.version.split()[0])


Project root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal3-controlled-QA-preflight-v1.0.2
Python: 3.11.9



## 1. Frozen Stage-B and frozen-A provenance gates

The Stage-B dose manifest is already sealed and cannot be altered by this notebook.  
The A registry and values are also immutable. This cell verifies both hashes before any waveform work occurs.


In [2]:

# ------------------------------------------------------------------
# Stage-B seal
# ------------------------------------------------------------------

STAGE_B = (
    ROOT
    / "outputs"
    / "goal3"
    / "stageB_signal_only_calibration_v1_3"
    / "final"
)

STAGE_B_SEAL = STAGE_B / "GOAL3_STAGE_B_SIGNAL_ONLY_SEAL.json"
DOSE_MANIFEST = STAGE_B / "tables" / "goal3_perturbation_manifest.csv"
AM_DISPOSITION = STAGE_B / "audit" / "AM_QADD_SIGNAL_ONLY_DISPOSITION.json"
QCHAN_AMENDMENT = STAGE_B / "audit" / "QCHAN_SIGNAL_ONLY_DESIGN_AMENDMENT.json"

for path in [
    STAGE_B_SEAL,
    DOSE_MANIFEST,
    AM_DISPOSITION,
    QCHAN_AMENDMENT,
]:
    if not path.exists():
        raise FileNotFoundError(path)

stage_b_seal = json.loads(STAGE_B_SEAL.read_text(encoding="utf-8"))

if stage_b_seal.get("status") != "GOAL3_STAGE_B_SIGNAL_ONLY_SEALED":
    raise RuntimeError("Goal 3 Stage B is not sealed.")

manifest_hash = sha256_file(DOSE_MANIFEST)

if manifest_hash != str(stage_b_seal["perturbation_manifest_sha256"]):
    raise RuntimeError("Stage-B perturbation manifest hash mismatch.")

dose_manifest = read_csv_checked(
    DOSE_MANIFEST,
    required=[
        "family",
        "transform",
        "dose_label",
        "candidate_value",
        "candidate_unit",
    ],
)

assert_no_clinical_columns("Stage-B dose manifest", dose_manifest)

if len(dose_manifest) != EXPECTED["sealed_dose_rows"]:
    raise RuntimeError(
        f"Expected 18 sealed dose rows, found {len(dose_manifest)}."
    )

if dose_manifest["transform"].nunique() != EXPECTED["sealed_transforms"]:
    raise RuntimeError("Expected six sealed perturbation structures.")

if dose_manifest["transform"].astype(str).str.contains(
    "amplitude_modulated", regex=False
).any():
    raise RuntimeError("Dropped AM-QADD unexpectedly appears in sealed manifest.")

if dose_manifest.groupby("transform")["dose_label"].nunique().ne(3).any():
    raise RuntimeError("Every sealed transform must have low/medium/high doses.")

# ------------------------------------------------------------------
# Frozen A
# ------------------------------------------------------------------

A_REGISTRY_PATH = MANIFESTS / "a_registry.csv"
A_VALUES_PATH = PROCESSED / "acoustic_features_frozen.csv"
A_FREEZE_MANIFEST_PATH = MANIFESTS / "a_freeze_manifest.json"

EXTRACTION_REGISTRY_PATH = (
    INTERIM
    / "acoustic_features"
    / "bamboo_acoustic_feature_registry.csv"
)

EXTRACTION_MANIFEST_PATH = (
    ROOT
    / "outputs"
    / "goal2"
    / "acoustic_extraction_v2_1"
    / "audit"
    / "bamboo_acoustic_extraction_manifest.json"
)

for path in [
    A_REGISTRY_PATH,
    A_VALUES_PATH,
    A_FREEZE_MANIFEST_PATH,
    EXTRACTION_REGISTRY_PATH,
    EXTRACTION_MANIFEST_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(path)

a_freeze_manifest = json.loads(
    A_FREEZE_MANIFEST_PATH.read_text(encoding="utf-8")
)
extraction_manifest = json.loads(
    EXTRACTION_MANIFEST_PATH.read_text(encoding="utf-8")
)

if a_freeze_manifest.get("status") != "FROZEN":
    raise RuntimeError("A freeze manifest is not FROZEN.")

if a_freeze_manifest.get("a_freeze_version") != A_FREEZE_VERSION:
    raise RuntimeError("Unexpected A freeze version.")

if a_freeze_manifest.get("acoustic_extraction_engine") != ACOUSTIC_ENGINE:
    raise RuntimeError("Unexpected frozen acoustic extraction engine.")

if extraction_manifest.get("engine_version") != ACOUSTIC_ENGINE:
    raise RuntimeError("Unexpected acoustic extraction manifest engine.")

if sha256_file(A_REGISTRY_PATH) != a_freeze_manifest["a_registry_sha256"]:
    raise RuntimeError("Frozen A registry hash mismatch.")

if sha256_file(A_VALUES_PATH) != a_freeze_manifest["acoustic_features_frozen_sha256"]:
    raise RuntimeError("Frozen A values hash mismatch.")

a_registry = read_csv_checked(
    A_REGISTRY_PATH,
    required=[
        "feature",
        "final_role",
        "final_transform",
        "support_indicator_column",
    ],
)
a_values = read_csv_checked(
    A_VALUES_PATH,
    required=["logical_recording_id", "participant_id"],
)
extraction_registry = read_csv_checked(
    EXTRACTION_REGISTRY_PATH,
    required=["feature", "acoustic_family"],
)

assert_no_clinical_columns("Frozen A registry", a_registry)
assert_no_clinical_columns("Frozen A values", a_values)

a_registry["feature"] = a_registry["feature"].astype(str)

PRIMARY_A = a_registry.loc[
    a_registry["final_role"].eq("primary"),
    "feature",
].tolist()

EXTENDED_A = a_registry.loc[
    a_registry["final_role"].eq("extended"),
    "feature",
].tolist()

RETAINED_A = PRIMARY_A + EXTENDED_A

if len(PRIMARY_A) != 6:
    raise RuntimeError(f"Expected six Primary-A features, found {len(PRIMARY_A)}.")

if len(EXTENDED_A) != 22:
    raise RuntimeError(f"Expected 22 Extended-A features, found {len(EXTENDED_A)}.")

if len(a_values) != EXPECTED["retained_recordings"]:
    raise RuntimeError("Frozen A values table is not 519 rows.")

print("FROZEN INPUT GATES: PASS")
print("Stage-B manifest SHA-256:", manifest_hash)
print("Primary-A:", len(PRIMARY_A))
print("Extended-A:", len(EXTENDED_A))
print("Frozen A rows:", len(a_values))


FROZEN INPUT GATES: PASS
Stage-B manifest SHA-256: c553dd58d978d3f5658c1c49a33595049f2a418c82c4e8af6c50d83aaba6c0e6
Primary-A: 6
Extended-A: 22
Frozen A rows: 519



## 2. Outcome-blind source set

Only the already-established 24-source signal-only pilot is read here.  
No diagnosis, age, sex, ALSFRS-R, model prediction, or model error is loaded.

The preflight uses the first two deterministic pilot ranks.


In [3]:

STAGE_A_CANDIDATES = [
    ROOT / "outputs" / "goal3" / "stageA_v1_1",
    ROOT / "outputs" / "goal3" / "stageA_v1_0",
]

STAGE_A = next(
    (
        path
        for path in STAGE_A_CANDIDATES
        if (path / "SUCCESS_STAGE_A.json").exists()
    ),
    None,
)

if STAGE_A is None:
    raise FileNotFoundError("No successful Goal 3 Stage-A directory found.")

PREP = ROOT / "outputs" / "goal3" / "stageB_signal_only_calibration_v1_0"

pilot = read_csv_checked(
    STAGE_A / "tables" / "goal3_signal_only_dose_pilot.csv",
    required=["participant_id", "logical_recording_id", "pilot_rank"],
)

crossfit = read_csv_checked(
    STAGE_A / "tables" / "goal3_fixed_fivefold_crossfit.csv",
    required=["participant_id", "outer_fold"],
)

media_audit = read_csv_checked(
    PREP / "audit" / "pilot_media_hash_audit.csv",
    required=[
        "participant_id",
        "logical_recording_id",
        "resolved_media_path",
        "observed_sha256",
        "sha256_matches",
    ],
)

for name, frame in [
    ("pilot", pilot),
    ("crossfit", crossfit),
    ("media_audit", media_audit),
]:
    assert_no_clinical_columns(name, frame)

pilot["participant_id"] = pilot["participant_id"].astype(str)
pilot["logical_recording_id"] = pilot["logical_recording_id"].astype(str)
crossfit["participant_id"] = crossfit["participant_id"].astype(str)
media_audit["participant_id"] = media_audit["participant_id"].astype(str)
media_audit["logical_recording_id"] = media_audit["logical_recording_id"].astype(str)

if len(pilot) != EXPECTED["pilot_sources"]:
    raise RuntimeError(f"Expected 24 pilot sources, found {len(pilot)}.")

pilot = (
    pilot.merge(
        crossfit,
        on="participant_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        media_audit[
            [
                "participant_id",
                "logical_recording_id",
                "resolved_media_path",
                "observed_sha256",
                "sha256_matches",
            ]
        ],
        on=["participant_id", "logical_recording_id"],
        how="left",
        validate="one_to_one",
    )
)

if pilot["outer_fold"].isna().any():
    raise RuntimeError("Pilot fold alignment failed.")

if not pilot["sha256_matches"].fillna(False).astype(bool).all():
    raise RuntimeError("Pilot media hash audit did not pass.")

preflight_sources = (
    pilot.sort_values("pilot_rank")
    .head(EXPECTED["preflight_sources"])
    .reset_index(drop=True)
)

print("OUTCOME-BLIND PREFLIGHT SOURCE GATE: PASS")
display(
    preflight_sources[
        [
            "pilot_rank",
            "participant_id",
            "logical_recording_id",
            "outer_fold",
        ]
    ]
)


OUTCOME-BLIND PREFLIGHT SOURCE GATE: PASS


,pilot_rank,participant_id,logical_recording_id,outer_fold
0,1,CAPT0000114,CAPT0000114_272_1_20240806_240_PSG_BAMBOO,3
1,2,CAPT0000099,CAPT0000099_272_1_20240523_240_PSG_BAMBOO,4



## 3. Exact pinned Paper-1 runtime and fixed QCHAN references

The same Paper-1 commit used by the prior Goal 3 calibration is required.  
QCHAN references are built exclusively from unmodified outer-training participants.


In [4]:

if not (EXTERNAL / ".git").exists():
    raise FileNotFoundError(
        f"Pinned Paper 1 repository is missing: {EXTERNAL}"
    )

observed_commit = subprocess.check_output(
    ["git", "-C", str(EXTERNAL), "rev-parse", "HEAD"],
    text=True,
).strip()

if observed_commit != PAPER1_COMMIT:
    raise RuntimeError(
        "Paper 1 commit mismatch.\n"
        f"Expected: {PAPER1_COMMIT}\n"
        f"Observed: {observed_commit}"
    )

paper1_src = EXTERNAL / "src"
if str(paper1_src) not in sys.path:
    sys.path.insert(0, str(paper1_src))

from paper1_qc.media import decode_native_audio
from paper1_qc.segmentation import (
    silero_speech_intervals,
    build_segmentation_views,
    load_silero_model,
)

from paper1_qc_reviewed.qadd_v420 import (
    extract_qadd,
    TimeInterval as QADDInterval,
)

from paper1_qc_reviewed.qgain_v410 import (
    extract_qgain,
    apply_gain_db as paper1_apply_gain_db,
    apply_level_envelope_db,
    TimeInterval as QGAINInterval,
)

from paper1_qc_reviewed.qrev_v400 import (
    extract_qrev,
    SpeechInterval as QREVInterval,
)

from paper1_qc_reviewed.qchan_v400 import (
    extract_recording_spectrum,
    compute_reference_relative_features,
    build_subject_balanced_loso_references,
    lowpass_filter as paper1_qchan_lowpass_filter,
    TimeInterval as QCHANInterval,
)

from paper1_qc_reviewed.qchan_v400_cohort import (
    load_recording_spectrum,
    save_reference_spectrum,
    load_reference_spectrum,
)

from paper1_qc import qdist_v410_candidate as qdist_detector
from paper1_qc_reviewed.qdist_v410_cohort import (
    inject_matched_hard_clip,
)

recording_identity = pd.read_csv(
    PROCESSED / "recording_table_phase0.csv",
    usecols=["participant_id", "logical_recording_id"],
    dtype=str,
)

if len(recording_identity) != EXPECTED["retained_recordings"]:
    raise RuntimeError("Recording identity table is not 519 rows.")

recording_identity = recording_identity.merge(
    crossfit,
    on="participant_id",
    how="left",
    validate="many_to_one",
)

qchan_manifest_path = MANIFESTS / "qchan_cache_ready.json"
qchan_manifest = json.loads(qchan_manifest_path.read_text(encoding="utf-8"))

if str(qchan_manifest.get("paper1_repo_commit", PAPER1_COMMIT)) != PAPER1_COMMIT:
    raise RuntimeError("QCHAN cache commit mismatch.")

QCHAN_INDEX = MANIFESTS / "qchan_spectrum_cache_index.csv"

if not QCHAN_INDEX.exists():
    raise FileNotFoundError(QCHAN_INDEX)

qchan_index = read_csv_checked(
    QCHAN_INDEX,
    required=["logical_recording_id"],
)
qchan_index["logical_recording_id"] = qchan_index["logical_recording_id"].astype(str)

path_col = (
    "cache_path"
    if "cache_path" in qchan_index.columns
    else "spectrum_path"
)

def resolve_cache_path(value):
    path = Path(str(value))
    if path.exists():
        return path.resolve()
    candidate = ROOT / path
    if candidate.exists():
        return candidate.resolve()
    raise FileNotFoundError(value)

qchan_index["resolved_cache_path"] = qchan_index[path_col].map(resolve_cache_path)

all_spectra = {}

for row in qchan_index.itertuples(index=False):
    rid = str(row.logical_recording_id)
    spectrum = load_recording_spectrum(Path(row.resolved_cache_path))
    if spectrum.logical_recording_id != rid:
        raise RuntimeError(f"QCHAN spectrum identity mismatch: {rid}")
    all_spectra[rid] = spectrum

if len(all_spectra) != 519:
    raise RuntimeError("Expected 519 cached QCHAN spectra.")

QCHAN_REF_DIR = CACHE / "qchan_fixed_outer_training_references"
QCHAN_REF_DIR.mkdir(parents=True, exist_ok=True)

fixed_qchan_references = {}
reference_audit_rows = []

for fold in sorted(crossfit["outer_fold"].unique()):
    fold = int(fold)
    cache_path = QCHAN_REF_DIR / f"outer_fold_{fold}.npz"

    train_participants = set(
        crossfit.loc[
            crossfit["outer_fold"].ne(fold),
            "participant_id",
        ].astype(str)
    )
    heldout_participants = set(
        crossfit.loc[
            crossfit["outer_fold"].eq(fold),
            "participant_id",
        ].astype(str)
    )

    training_rows = recording_identity.loc[
        recording_identity["participant_id"].isin(train_participants),
        ["logical_recording_id", "participant_id"],
    ].copy()

    training_rows = training_rows.rename(
        columns={"participant_id": "subject_id"}
    )
    training_rows["task_stratum"] = "BAMBOO_PASSAGE"

    training_spectra = {
        rid: all_spectra[rid]
        for rid in training_rows["logical_recording_id"].astype(str)
    }

    if cache_path.exists():
        reference = load_reference_spectrum(cache_path)
        reused = True
    else:
        dummy_id = f"__GOAL3_OUTER_FOLD_{fold}_REFERENCE_TARGET__"
        dummy_subject = f"__GOAL3_HELDOUT_FOLD_{fold}__"

        metadata = pd.concat(
            [
                training_rows,
                pd.DataFrame(
                    [{
                        "logical_recording_id": dummy_id,
                        "subject_id": dummy_subject,
                        "task_stratum": "BAMBOO_PASSAGE",
                    }]
                ),
            ],
            ignore_index=True,
        )

        references = build_subject_balanced_loso_references(
            training_spectra,
            metadata,
        )
        reference = references[dummy_id]
        save_reference_spectrum(reference, cache_path)
        reused = False

    member_subjects = set(map(str, reference.member_subject_ids))

    if member_subjects & heldout_participants:
        raise RuntimeError(
            f"QCHAN fold {fold} reference leaks held-out participants."
        )

    if not member_subjects.issubset(train_participants):
        raise RuntimeError(
            f"QCHAN fold {fold} reference contains non-training participants."
        )

    if reference.status != "measured":
        raise RuntimeError(
            f"QCHAN fold {fold} reference status is {reference.status!r}."
        )

    fixed_qchan_references[fold] = reference

    reference_audit_rows.append({
        "outer_fold": fold,
        "reference_recording_count": reference.recording_count,
        "reference_subject_count": reference.subject_count,
        "reference_sha256": reference.reference_sha256,
        "checkpoint_reused": reused,
        "heldout_subject_overlap_n": 0,
    })

reference_audit = pd.DataFrame(reference_audit_rows)
atomic_csv(reference_audit, AUDIT / "qchan_fixed_reference_audit.csv")

print("PINNED PAPER 1 + FIXED QCHAN REFERENCES: PASS")
display(reference_audit)


PINNED PAPER 1 + FIXED QCHAN REFERENCES: PASS


,outer_fold,reference_recording_count,reference_subject_count,reference_sha256,checkpoint_reused,heldout_subject_overlap_n
0,1,412,179,5db7994c3231f6a8b71f5315146a38b4eb6bcadfe8e4da...,True,0
1,2,411,179,f885d06f99be45bab55f69b7dc87e7d956a89b3e73fb0b...,True,0
2,3,414,179,f231f2f2da998d0bd23bb0bfac35a3f721f54050a23cbe...,True,0
3,4,421,179,f6d51ef0213bda534387c7bd161dcb7c8d4cef58be4300...,True,0
4,5,418,180,07c83534652742fe2aaa44b05a2f0c81a6c8e6a761f8ec...,True,0



## 4. Exact frozen v2.1 acoustic feature implementation

The following timing, pitch/harmonicity, formant, and envelope-rhythm functions are copied directly from the supplied v2.1 acoustic extraction notebook.  
The only adaptation is the wrapper: instead of reading the previously frozen segmentation table, it accepts the **fresh Goal 3 re-segmentation views** produced from the current waveform.


In [5]:

# Exact constants from the frozen v2.1 Bamboo acoustic extractor.

TARGET_SR = 16_000
BAMBOO_WORD_COUNT = 99
BAMBOO_SYLLABLE_COUNT = 137
MIN_INTERNAL_PAUSE_SEC = 0.300

F0_MIN_HZ = 60.0
F0_MAX_HZ = 400.0
F0_FRAME_SEC = 0.040
F0_HOP_SEC = 0.020
F0_MIN_AUTOCORR = 0.35
MAX_PITCH_FRAMES = 800

FORMANT_FRAME_SEC = 0.030
FORMANT_HOP_SEC = 0.010
MAX_FORMANT_FRAMES = 400
LPC_ORDER = 14
PREEMPHASIS = 0.97
FORMANT_MIN_PERIODICITY = 0.30
F1_RANGE_HZ = (180.0, 1200.0)
F2_RANGE_HZ = (600.0, 3500.0)
MAX_FORMANT_BANDWIDTH_HZ = 500.0
MIN_VALID_FORMANT_FRAMES = 30

ENVELOPE_TARGET_HZ = 100
ENVELOPE_MAX_ANALYSIS_HZ = 10.0

registry = extraction_registry.copy()
FEATURES = registry["feature"].astype(str).tolist()
SUPPORT_COLUMNS = [f"{feature}__supported" for feature in FEATURES]

if len(FEATURES) != int(extraction_manifest["n_implemented_candidate_features"]):
    raise RuntimeError("Implemented feature count differs from extraction manifest.")

print("Frozen v2.1 implemented candidates:", len(FEATURES))


Frozen v2.1 implemented candidates: 38


### SPA-compatible timing implementation

In [6]:
def merge_overlapping_intervals(rows: pd.DataFrame):
    raw = sorted(
        [
            (float(r.start_sec), float(r.end_sec))
            for r in rows.itertuples(index=False)
            if (
                np.isfinite(r.start_sec)
                and np.isfinite(r.end_sec)
                and r.end_sec > r.start_sec
            )
        ]
    )

    if not raw:
        return []

    merged = [list(raw[0])]

    for start, end in raw[1:]:
        if start <= merged[-1][1] + 1e-9:
            merged[-1][1] = max(
                merged[-1][1],
                end,
            )
        else:
            merged.append([start, end])

    return [
        (float(start), float(end))
        for start, end in merged
    ]


def primary_intervals_for(recording_id):
    rows = primary_intervals.loc[
        primary_intervals[
            "logical_recording_id"
        ].eq(str(recording_id))
    ].sort_values(["start_sec", "end_sec"])

    return merge_overlapping_intervals(rows)


def phrase_intervals_300ms(interval_list):
    """
    Merge adjacent primary-speech intervals across gaps <300 ms.

    The resulting phrase is therefore bounded only by clinically
    qualifying pauses >=300 ms.
    """
    if not interval_list:
        return []

    phrases = [
        [float(interval_list[0][0]),
         float(interval_list[0][1])]
    ]

    for start, end in interval_list[1:]:
        gap = float(start - phrases[-1][1])

        if gap < MIN_INTERNAL_PAUSE_SEC:
            phrases[-1][1] = float(end)
        else:
            phrases.append(
                [float(start), float(end)]
            )

    return [
        (float(start), float(end))
        for start, end in phrases
    ]


def safe_iqr(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 2:
        return np.nan

    return float(
        np.quantile(x, 0.75)
        - np.quantile(x, 0.25)
    )


def safe_cv(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) < 2:
        return np.nan

    mean = float(np.mean(x))

    if not np.isfinite(mean) or abs(mean) <= 1e-12:
        return np.nan

    return float(
        np.std(x, ddof=1) / mean
    )


def _event_duration_by_half(
    intervals,
    passage_start,
    passage_end,
):
    midpoint = (
        float(passage_start)
        + float(passage_end)
    ) / 2.0

    first = []
    second = []

    for start, end in intervals:
        event_mid = (start + end) / 2.0

        if event_mid < midpoint:
            first.append(end - start)
        else:
            second.append(end - start)

    return first, second


def timing_features_for(interval_list):
    family_features = registry.loc[
        registry["acoustic_family"].eq(
            "timing_fluency"
        ),
        "feature",
    ].tolist()

    out = {
        feature: np.nan
        for feature in family_features
    }

    if not interval_list:
        return out

    starts = np.array(
        [a for a, _ in interval_list],
        dtype=float,
    )
    ends = np.array(
        [b for _, b in interval_list],
        dtype=float,
    )

    passage_start = float(starts[0])
    passage_end = float(ends[-1])
    span = passage_end - passage_start

    raw_speech_support = float(
        np.sum(ends - starts)
    )

    if span <= 0 or raw_speech_support <= 0:
        return out

    gaps = starts[1:] - ends[:-1]
    qualifying_pauses = gaps[
        gaps >= MIN_INTERNAL_PAUSE_SEC
    ]

    pause_count = int(len(qualifying_pauses))
    total_pause = (
        float(np.sum(qualifying_pauses))
        if pause_count
        else 0.0
    )

    percent_pause = 100.0 * total_pause / span
    articulation_time = span - total_pause

    phrases = phrase_intervals_300ms(
        interval_list
    )
    phrase_durations = np.asarray(
        [end - start for start, end in phrases],
        dtype=float,
    )

    out["bamboo_passage_span_sec"] = span
    out["bamboo_primary_speech_support_sec"] = (
        raw_speech_support
    )
    out["bamboo_primary_speech_fraction"] = float(
        np.clip(raw_speech_support / span, 0.0, 1.0)
    )

    out["bamboo_pause_count_300ms"] = float(
        pause_count
    )
    out["bamboo_total_pause_sec_300ms"] = (
        total_pause
    )
    out["bamboo_percent_pause_time_300ms"] = (
        percent_pause
    )
    out["bamboo_pause_rate_per_min_300ms"] = (
        pause_count / (span / 60.0)
    )

    if pause_count >= 1:
        out["bamboo_pause_mean_sec_300ms"] = float(
            np.mean(qualifying_pauses)
        )
        out["bamboo_pause_median_sec_300ms"] = float(
            np.median(qualifying_pauses)
        )

    out["bamboo_pause_iqr_sec_300ms"] = (
        safe_iqr(qualifying_pauses)
    )
    out["bamboo_pause_cv_300ms"] = (
        safe_cv(qualifying_pauses)
    )

    out["bamboo_phrase_count_300ms"] = float(
        len(phrase_durations)
    )

    if len(phrase_durations):
        out["bamboo_phrase_mean_sec_300ms"] = float(
            np.mean(phrase_durations)
        )
        out["bamboo_phrase_median_sec_300ms"] = float(
            np.median(phrase_durations)
        )

    out["bamboo_phrase_iqr_sec_300ms"] = (
        safe_iqr(phrase_durations)
    )
    out["bamboo_phrase_cv_300ms"] = (
        safe_cv(phrase_durations)
    )

    out["bamboo_nominal_speaking_rate_wpm"] = (
        BAMBOO_WORD_COUNT * 60.0 / span
    )

    if articulation_time > 0:
        out[
            "bamboo_nominal_articulation_rate_wpm"
        ] = (
            BAMBOO_WORD_COUNT
            * 60.0
            / articulation_time
        )

        out[
            "bamboo_nominal_articulation_rate_syll_per_sec"
        ] = (
            BAMBOO_SYLLABLE_COUNT
            / articulation_time
        )

    # ----------------------------------------------------------
    # Within-passage half comparisons
    # ----------------------------------------------------------

    midpoint = (passage_start + passage_end) / 2.0

    first_half_duration = midpoint - passage_start
    second_half_duration = passage_end - midpoint

    # Qualifying pause intervals reconstructed from adjacent
    # primary-speech boundaries.
    pause_intervals = [
        (ends[i], starts[i + 1])
        for i in range(len(starts) - 1)
        if (
            starts[i + 1] - ends[i]
            >= MIN_INTERNAL_PAUSE_SEC
        )
    ]

    first_pause = 0.0
    second_pause = 0.0

    for start, end in pause_intervals:
        # Split a pause if it crosses the midpoint.
        first_pause += max(
            0.0,
            min(end, midpoint)
            - max(start, passage_start),
        )
        second_pause += max(
            0.0,
            min(end, passage_end)
            - max(start, midpoint),
        )

    if first_half_duration > 0 and second_half_duration > 0:
        first_pct = (
            100.0
            * first_pause
            / first_half_duration
        )
        second_pct = (
            100.0
            * second_pause
            / second_half_duration
        )

        out[
            "bamboo_pause_fraction_last_minus_first_half_pct"
        ] = second_pct - first_pct

    first_phrase, second_phrase = (
        _event_duration_by_half(
            phrases,
            passage_start,
            passage_end,
        )
    )

    if first_phrase and second_phrase:
        out[
            "bamboo_phrase_duration_last_minus_first_half_sec"
        ] = (
            float(np.median(second_phrase))
            - float(np.median(first_phrase))
        )

    return out


# --------------------------------------------------------------
# Deterministic timing tests
# --------------------------------------------------------------

_test_intervals = [
    (1.0, 2.0),
    (2.2, 3.0),  # 200-ms gap -> same phrase
    (3.5, 4.5),  # 500-ms qualifying pause
    (4.6, 5.0),  # 100-ms gap -> same phrase
]

_test = timing_features_for(
    _test_intervals
)

assert np.isclose(
    _test["bamboo_passage_span_sec"],
    4.0,
)

assert _test[
    "bamboo_pause_count_300ms"
] == 1

assert np.isclose(
    _test["bamboo_total_pause_sec_300ms"],
    0.5,
)

assert np.isclose(
    _test["bamboo_percent_pause_time_300ms"],
    12.5,
)

# Two clinically defined phrases:
# [1.0, 3.0] and [3.5, 5.0]
assert _test[
    "bamboo_phrase_count_300ms"
] == 2

# Articulation time = 4.0 - 0.5 = 3.5 sec
assert np.isclose(
    _test[
        "bamboo_nominal_articulation_rate_syll_per_sec"
    ],
    BAMBOO_SYLLABLE_COUNT / 3.5,
)

print("SPA-COMPATIBLE TIMING UTILITIES: PASS")


SPA-COMPATIBLE TIMING UTILITIES: PASS


### Pitch and harmonicity implementation

In [7]:
def extract_frames_with_times(
    waveform,
    sr,
    interval_list,
    frame_sec,
    hop_sec,
    max_frames,
):
    frame_n = int(round(frame_sec * sr))
    hop_n = int(round(hop_sec * sr))

    frames = []
    mid_times = []

    for start_sec, end_sec in interval_list:
        lo = max(
            0,
            int(round(start_sec * sr)),
        )
        hi = min(
            len(waveform),
            int(round(end_sec * sr)),
        )

        if hi - lo < frame_n:
            continue

        starts = np.arange(
            lo,
            hi - frame_n + 1,
            hop_n,
            dtype=int,
        )

        for start in starts:
            frames.append(
                waveform[start:start + frame_n]
            )
            mid_times.append(
                (start + frame_n / 2)
                / sr
            )

    if not frames:
        return (
            np.empty((0, frame_n), dtype=float),
            np.array([], dtype=float),
        )

    if len(frames) > max_frames:
        keep = np.unique(
            np.linspace(
                0,
                len(frames) - 1,
                max_frames,
            ).round().astype(int)
        )

        frames = [
            frames[i]
            for i in keep
        ]
        mid_times = [
            mid_times[i]
            for i in keep
        ]

    return (
        np.asarray(frames, dtype=float),
        np.asarray(mid_times, dtype=float),
    )


def _quadratic_peak_offset(y_left, y_mid, y_right):
    denominator = (
        y_left
        - 2.0 * y_mid
        + y_right
    )

    if abs(denominator) <= 1e-12:
        return 0.0

    delta = (
        0.5
        * (y_left - y_right)
        / denominator
    )

    return float(
        np.clip(delta, -1.0, 1.0)
    )


def pitch_harmonicity_from_frames(
    frames,
    frame_times,
    sr,
):
    n_input = len(frames)

    if n_input == 0:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            0,
        )

    X = (
        np.asarray(frames, dtype=float)
        - np.mean(
            frames,
            axis=1,
            keepdims=True,
        )
    )

    X = (
        X
        * np.hanning(X.shape[1])[None, :]
    )

    rms = np.sqrt(
        np.mean(X * X, axis=1)
    )

    finite_positive = rms[
        np.isfinite(rms)
        & (rms > 0)
    ]

    if len(finite_positive) == 0:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            n_input,
        )

    rms_floor = max(
        np.finfo(float).eps,
        float(
            np.quantile(
                finite_positive,
                0.10,
            )
        ) * 0.35,
    )

    active_mask = rms > rms_floor
    X_active = X[active_mask]
    t_active = frame_times[active_mask]

    if len(X_active) == 0:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            n_input,
        )

    n = X_active.shape[1]
    nfft = (
        1
        << int(
            np.ceil(
                np.log2(2 * n - 1)
            )
        )
    )

    spectrum = np.fft.rfft(
        X_active,
        n=nfft,
        axis=1,
    )

    autocorr = np.fft.irfft(
        spectrum
        * np.conj(spectrum),
        n=nfft,
        axis=1,
    )[:, :n]

    energy = autocorr[:, 0]

    good = (
        np.isfinite(energy)
        & (energy > np.finfo(float).eps)
    )

    autocorr = autocorr[good]
    t_active = t_active[good]
    energy = energy[good]

    if len(autocorr) == 0:
        return (
            np.array([]),
            np.array([]),
            np.array([]),
            n_input,
        )

    normalized = (
        autocorr
        / energy[:, None]
    )

    min_lag = max(
        1,
        int(
            np.floor(
                sr / F0_MAX_HZ
            )
        ),
    )

    max_lag = min(
        n - 2,
        int(
            np.ceil(
                sr / F0_MIN_HZ
            )
        ),
    )

    search = normalized[
        :,
        min_lag:max_lag + 1,
    ]

    f0_values = []
    harmonicity_values = []
    voiced_times = []

    for row, time_value in zip(
        search,
        t_active,
    ):
        peaks, _ = signal.find_peaks(row)

        if len(peaks):
            best = int(
                peaks[
                    np.argmax(row[peaks])
                ]
            )
        else:
            best = int(np.argmax(row))

        peak_r = float(row[best])

        if (
            not np.isfinite(peak_r)
            or peak_r < F0_MIN_AUTOCORR
        ):
            continue

        delta = 0.0

        if 0 < best < len(row) - 1:
            delta = _quadratic_peak_offset(
                row[best - 1],
                row[best],
                row[best + 1],
            )

        lag = (
            min_lag
            + best
            + delta
        )

        if lag <= 0:
            continue

        f0 = float(sr / lag)

        if not (
            F0_MIN_HZ
            <= f0
            <= F0_MAX_HZ
        ):
            continue

        rr = float(
            np.clip(
                peak_r,
                1e-6,
                1 - 1e-6,
            )
        )

        harmonicity = (
            10.0
            * np.log10(
                rr / (1.0 - rr)
            )
        )

        f0_values.append(f0)
        harmonicity_values.append(
            harmonicity
        )
        voiced_times.append(
            float(time_value)
        )

    return (
        np.asarray(f0_values, dtype=float),
        np.asarray(
            harmonicity_values,
            dtype=float,
        ),
        np.asarray(
            voiced_times,
            dtype=float,
        ),
        n_input,
    )


def _linear_slope_per_minute(
    values,
    times,
):
    values = np.asarray(
        values,
        dtype=float,
    )
    times = np.asarray(
        times,
        dtype=float,
    )

    keep = (
        np.isfinite(values)
        & np.isfinite(times)
    )

    values = values[keep]
    times = times[keep]

    if len(values) < 10:
        return np.nan

    centered_time = (
        times - np.mean(times)
    )

    denominator = float(
        np.sum(
            centered_time ** 2
        )
    )

    if denominator <= 0:
        return np.nan

    slope_per_sec = float(
        np.sum(
            centered_time
            * (
                values
                - np.mean(values)
            )
        )
        / denominator
    )

    return slope_per_sec * 60.0


def pitch_summary(
    waveform,
    sr,
    interval_list,
):
    features = {
        feature: np.nan
        for feature in registry.loc[
            registry[
                "acoustic_family"
            ].eq("phonation_prosody"),
            "feature",
        ]
    }

    frames, times = (
        extract_frames_with_times(
            waveform,
            sr,
            interval_list,
            F0_FRAME_SEC,
            F0_HOP_SEC,
            MAX_PITCH_FRAMES,
        )
    )

    (
        f0,
        harmonicity,
        voiced_times,
        n_analyzed,
    ) = pitch_harmonicity_from_frames(
        frames,
        times,
        sr,
    )

    diagnostics = {
        "pitch_frames_analyzed": int(
            n_analyzed
        ),
        "pitch_frames_voiced": int(
            len(f0)
        ),
    }

    if n_analyzed < 20:
        return features, diagnostics

    features[
        "bamboo_voiced_frame_fraction"
    ] = float(
        len(f0) / n_analyzed
    )

    if len(f0) < 10:
        return features, diagnostics

    f0_semitones = (
        12.0
        * np.log2(f0)
    )

    features[
        "bamboo_f0_median_hz"
    ] = float(np.median(f0))

    features[
        "bamboo_f0_iqr_semitones"
    ] = safe_iqr(f0_semitones)

    features[
        "bamboo_f0_p10_p90_range_semitones"
    ] = float(
        np.quantile(
            f0_semitones,
            0.90,
        )
        - np.quantile(
            f0_semitones,
            0.10,
        )
    )

    features[
        "bamboo_f0_slope_semitones_per_min"
    ] = _linear_slope_per_minute(
        f0_semitones,
        voiced_times,
    )

    features[
        "bamboo_harmonicity_median_db"
    ] = float(
        np.median(harmonicity)
    )

    features[
        "bamboo_harmonicity_iqr_db"
    ] = safe_iqr(harmonicity)

    passage_midpoint = (
        interval_list[0][0]
        + interval_list[-1][1]
    ) / 2.0

    first = (
        voiced_times
        < passage_midpoint
    )
    second = ~first

    if first.sum() >= 5 and second.sum() >= 5:
        features[
            "bamboo_f0_last_minus_first_half_semitones"
        ] = (
            float(
                np.median(
                    f0_semitones[second]
                )
            )
            - float(
                np.median(
                    f0_semitones[first]
                )
            )
        )

        features[
            "bamboo_harmonicity_last_minus_first_half_db"
        ] = (
            float(
                np.median(
                    harmonicity[second]
                )
            )
            - float(
                np.median(
                    harmonicity[first]
                )
            )
        )

    return features, diagnostics


# --------------------------------------------------------------
# Synthetic F0 validation
# --------------------------------------------------------------

sr_test = 16_000
t = (
    np.arange(
        int(2.0 * sr_test)
    )
    / sr_test
)

for target_hz in [
    100.0,
    180.0,
    250.0,
    320.0,
]:
    y = (
        np.sin(
            2 * np.pi
            * target_hz
            * t
        )
        + 0.35
        * np.sin(
            2 * np.pi
            * 2 * target_hz
            * t
        )
        + 0.15
        * np.sin(
            2 * np.pi
            * 3 * target_hz
            * t
        )
    )

    features_test, _ = (
        pitch_summary(
            y,
            sr_test,
            [(0.0, 2.0)],
        )
    )

    observed = features_test[
        "bamboo_f0_median_hz"
    ]

    if (
        not np.isfinite(observed)
        or abs(
            observed - target_hz
        ) / target_hz > 0.05
    ):
        raise RuntimeError(
            "Pitch synthetic validation failed: "
            f"target={target_hz}, "
            f"observed={observed}"
        )

print(
    "PITCH / HARMONICITY "
    "SYNTHETIC VALIDATION: PASS"
)


PITCH / HARMONICITY SYNTHETIC VALIDATION: PASS


### Formant implementation

In [8]:
def _frame_periodicity(
    frame,
    sr,
):
    x = np.asarray(
        frame,
        dtype=float,
    )

    if len(x) < 8:
        return np.nan

    x = x - np.mean(x)
    x = x * np.hanning(len(x))

    energy = float(
        np.dot(x, x)
    )

    if (
        not np.isfinite(energy)
        or energy <= 1e-12
    ):
        return np.nan

    autocorr = signal.fftconvolve(
        x,
        x[::-1],
        mode="full",
    )[len(x) - 1:]

    autocorr = autocorr / energy

    min_lag = max(
        1,
        int(
            np.floor(
                sr / F0_MAX_HZ
            )
        ),
    )

    max_lag = min(
        len(x) - 2,
        int(
            np.ceil(
                sr / F0_MIN_HZ
            )
        ),
    )

    if max_lag <= min_lag:
        return np.nan

    return float(
        np.max(
            autocorr[
                min_lag:max_lag + 1
            ]
        )
    )


def _lpc_f1_f2(
    frame,
    sr,
):
    x = np.asarray(
        frame,
        dtype=float,
    )

    if (
        len(x) < LPC_ORDER + 4
        or not np.isfinite(x).all()
    ):
        return np.nan, np.nan

    x = x - np.mean(x)

    rms = float(
        np.sqrt(
            np.mean(x * x)
        )
    )

    if rms <= 1e-8:
        return np.nan, np.nan

    x = signal.lfilter(
        [1.0, -PREEMPHASIS],
        [1.0],
        x,
    )

    x = x * np.hamming(len(x))

    full = np.correlate(
        x,
        x,
        mode="full",
    )

    center = len(x) - 1

    r = full[
        center:
        center + LPC_ORDER + 1
    ].astype(float)

    if (
        len(r) != LPC_ORDER + 1
        or not np.isfinite(r).all()
        or r[0] <= 1e-12
    ):
        return np.nan, np.nan

    # Very small diagonal loading for numerical stability.
    r[0] = (
        r[0]
        * (1.0 + 1e-6)
    )

    try:
        a_rest = linalg.solve_toeplitz(
            (
                r[:LPC_ORDER],
                r[:LPC_ORDER],
            ),
            -r[1:LPC_ORDER + 1],
            check_finite=False,
        )
    except Exception:
        return np.nan, np.nan

    coefficients = np.r_[
        1.0,
        a_rest,
    ]

    roots = np.roots(coefficients)

    roots = roots[
        np.imag(roots) > 0.01
    ]

    if len(roots) == 0:
        return np.nan, np.nan

    frequencies = (
        np.angle(roots)
        * sr
        / (2.0 * np.pi)
    )

    bandwidths = (
        -0.5
        * (
            sr
            / (2.0 * np.pi)
        )
        * np.log(
            np.abs(roots) ** 2
        )
    )

    valid = (
        np.isfinite(frequencies)
        & np.isfinite(bandwidths)
        & (frequencies >= 150.0)
        & (frequencies <= 4000.0)
        & (bandwidths > 0.0)
        & (
            bandwidths
            <= MAX_FORMANT_BANDWIDTH_HZ
        )
    )

    candidate_freqs = sorted(
        [
            float(value)
            for value in frequencies[valid]
        ]
    )

    # Suppress very close duplicate resonances.
    deduplicated = []

    for frequency in candidate_freqs:
        if (
            not deduplicated
            or (
                frequency
                - deduplicated[-1]
                > 120.0
            )
        ):
            deduplicated.append(
                frequency
            )

    f1_candidates = [
        f
        for f in deduplicated
        if (
            F1_RANGE_HZ[0]
            <= f
            <= F1_RANGE_HZ[1]
        )
    ]

    if not f1_candidates:
        return np.nan, np.nan

    f1 = f1_candidates[0]

    f2_candidates = [
        f
        for f in deduplicated
        if (
            max(
                F2_RANGE_HZ[0],
                f1 + 250.0,
            )
            <= f
            <= F2_RANGE_HZ[1]
        )
    ]

    if not f2_candidates:
        return f1, np.nan

    return (
        float(f1),
        float(f2_candidates[0]),
    )


def formant_summary(
    waveform,
    sr,
    interval_list,
):
    features = {
        feature: np.nan
        for feature in registry.loc[
            registry[
                "acoustic_family"
            ].eq("segmental_articulation"),
            "feature",
        ]
    }

    frames, _ = (
        extract_frames_with_times(
            waveform,
            sr,
            interval_list,
            FORMANT_FRAME_SEC,
            FORMANT_HOP_SEC,
            MAX_FORMANT_FRAMES,
        )
    )

    n_input = len(frames)

    if n_input == 0:
        return features, {
            "formant_frames_analyzed": 0,
            "formant_frames_valid": 0,
            "formant_valid_frame_fraction": np.nan,
        }

    rms = np.sqrt(
        np.mean(
            (
                frames
                - np.mean(
                    frames,
                    axis=1,
                    keepdims=True,
                )
            ) ** 2,
            axis=1,
        )
    )

    finite_positive = rms[
        np.isfinite(rms)
        & (rms > 0)
    ]

    if len(finite_positive) == 0:
        return features, {
            "formant_frames_analyzed": n_input,
            "formant_frames_valid": 0,
            "formant_valid_frame_fraction": 0.0,
        }

    rms_floor = max(
        np.finfo(float).eps,
        float(
            np.quantile(
                finite_positive,
                0.10,
            )
        ) * 0.35,
    )

    f1_values = []
    f2_values = []

    for frame, frame_rms in zip(
        frames,
        rms,
    ):
        if (
            not np.isfinite(frame_rms)
            or frame_rms <= rms_floor
        ):
            continue

        periodicity = _frame_periodicity(
            frame,
            sr,
        )

        if (
            not np.isfinite(periodicity)
            or periodicity
            < FORMANT_MIN_PERIODICITY
        ):
            continue

        f1, f2 = _lpc_f1_f2(
            frame,
            sr,
        )

        if (
            np.isfinite(f1)
            and np.isfinite(f2)
        ):
            f1_values.append(f1)
            f2_values.append(f2)

    f1_values = np.asarray(
        f1_values,
        dtype=float,
    )
    f2_values = np.asarray(
        f2_values,
        dtype=float,
    )

    n_valid = len(f2_values)

    diagnostics = {
        "formant_frames_analyzed": int(
            n_input
        ),
        "formant_frames_valid": int(
            n_valid
        ),
        "formant_valid_frame_fraction": (
            float(n_valid / n_input)
            if n_input
            else np.nan
        ),
    }

    if n_valid < MIN_VALID_FORMANT_FRAMES:
        return features, diagnostics

    features["bamboo_f1_iqr_hz"] = (
        safe_iqr(f1_values)
    )

    features[
        "bamboo_f1_p10_p90_range_hz"
    ] = float(
        np.quantile(
            f1_values,
            0.90,
        )
        - np.quantile(
            f1_values,
            0.10,
        )
    )

    features["bamboo_f2_iqr_hz"] = (
        safe_iqr(f2_values)
    )

    features[
        "bamboo_f2_p10_p90_range_hz"
    ] = float(
        np.quantile(
            f2_values,
            0.90,
        )
        - np.quantile(
            f2_values,
            0.10,
        )
    )

    return features, diagnostics


# --------------------------------------------------------------
# Synthetic all-pole vowel validation
# --------------------------------------------------------------

def _synthesize_vowel(
    f0,
    formants,
    bandwidths,
    duration=1.5,
    sr=16_000,
):
    n = int(
        round(duration * sr)
    )

    excitation = np.zeros(
        n,
        dtype=float,
    )

    period = max(
        1,
        int(round(sr / f0)),
    )

    excitation[::period] = 1.0

    y = excitation.copy()

    for frequency, bandwidth in zip(
        formants,
        bandwidths,
    ):
        radius = np.exp(
            -np.pi
            * bandwidth
            / sr
        )

        theta = (
            2.0
            * np.pi
            * frequency
            / sr
        )

        denominator = [
            1.0,
            -2.0
            * radius
            * np.cos(theta),
            radius ** 2,
        ]

        y = signal.lfilter(
            [1.0],
            denominator,
            y,
        )

    peak = float(
        np.max(np.abs(y))
    )

    if peak > 0:
        y = y / peak

    return y


synthetic_vowels = [
    ("i", 120.0, (300.0, 2300.0, 3000.0)),
    ("u", 120.0, (350.0, 900.0, 2200.0)),
    ("a", 120.0, (700.0, 1200.0, 2500.0)),
]

for label, f0, target_formants in synthetic_vowels:
    y = _synthesize_vowel(
        f0=f0,
        formants=target_formants,
        bandwidths=(
            70.0,
            100.0,
            150.0,
        ),
    )

    frames_test, _ = (
        extract_frames_with_times(
            y,
            16_000,
            [(0.1, 1.4)],
            FORMANT_FRAME_SEC,
            FORMANT_HOP_SEC,
            100,
        )
    )

    estimates = np.asarray(
        [
            _lpc_f1_f2(
                frame,
                16_000,
            )
            for frame in frames_test
        ],
        dtype=float,
    )

    observed_f1 = float(
        np.nanmedian(
            estimates[:, 0]
        )
    )

    observed_f2 = float(
        np.nanmedian(
            estimates[:, 1]
        )
    )

    target_f1 = target_formants[0]
    target_f2 = target_formants[1]

    if (
        not np.isfinite(observed_f1)
        or not np.isfinite(observed_f2)
        or abs(
            observed_f1 - target_f1
        ) > 120.0
        or abs(
            observed_f2 - target_f2
        ) > 180.0
    ):
        raise RuntimeError(
            "Formant synthetic validation failed for "
            f"/{label}/: target F1/F2="
            f"{target_f1:.0f}/{target_f2:.0f}, "
            f"observed="
            f"{observed_f1:.1f}/{observed_f2:.1f}"
        )

print(
    "FORMANT SYNTHETIC VALIDATION: PASS"
)


FORMANT SYNTHETIC VALIDATION: PASS


### Envelope-rhythm implementation

In [9]:
def envelope_rhythm_summary(
    waveform,
    sr,
    interval_list,
):
    features = {
        feature: np.nan
        for feature in registry.loc[
            registry[
                "acoustic_family"
            ].eq("rhythm_envelope"),
            "feature",
        ]
    }

    if not interval_list:
        return features, {
            "envelope_support_sec": 0.0
        }

    start = max(
        0,
        int(
            round(
                interval_list[0][0]
                * sr
            )
        ),
    )

    end = min(
        len(waveform),
        int(
            round(
                interval_list[-1][1]
                * sr
            )
        ),
    )

    x = np.asarray(
        waveform[start:end],
        dtype=float,
    )

    duration = len(x) / sr

    diagnostics = {
        "envelope_support_sec": float(
            duration
        )
    }

    if duration < 5.0 or len(x) < 128:
        return features, diagnostics

    x = x - np.mean(x)

    envelope = np.abs(
        signal.hilbert(x)
    )

    positive = envelope[
        np.isfinite(envelope)
        & (envelope > 0)
    ]

    if len(positive) < 100:
        return features, diagnostics

    scale = float(
        np.median(positive)
    )

    if (
        not np.isfinite(scale)
        or scale <= np.finfo(float).eps
    ):
        return features, diagnostics

    envelope = envelope / scale

    clip_high = float(
        np.quantile(
            envelope[
                np.isfinite(envelope)
            ],
            0.995,
        )
    )

    if (
        not np.isfinite(clip_high)
        or clip_high <= 0
    ):
        return features, diagnostics

    envelope = np.clip(
        envelope,
        0.0,
        clip_high,
    )

    gcd_value = math.gcd(
        int(sr),
        int(ENVELOPE_TARGET_HZ),
    )

    env_ds = signal.resample_poly(
        envelope,
        ENVELOPE_TARGET_HZ
        // gcd_value,
        sr // gcd_value,
    )

    env_ds = np.asarray(
        env_ds,
        dtype=float,
    )

    if not np.isfinite(env_ds).all():
        return features, diagnostics

    env_ds = env_ds - np.mean(env_ds)

    if len(env_ds) < ENVELOPE_TARGET_HZ * 5:
        return features, diagnostics

    nperseg = min(
        len(env_ds),
        1024,
    )

    frequency, psd = signal.welch(
        env_ds,
        fs=ENVELOPE_TARGET_HZ,
        nperseg=nperseg,
        detrend="constant",
        scaling="density",
    )

    band = (
        (frequency >= 0.5)
        & (
            frequency
            <= ENVELOPE_MAX_ANALYSIS_HZ
        )
    )

    if band.sum() < 5:
        return features, diagnostics

    f = frequency[band]
    p = np.maximum(
        psd[band],
        0.0,
    )

    total = float(
        np.sum(p)
    )

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        return features, diagnostics

    peak_band = (
        (f >= 0.5)
        & (f <= 8.0)
    )

    features[
        "bamboo_envelope_mod_peak_hz"
    ] = float(
        f[peak_band][
            np.argmax(
                p[peak_band]
            )
        ]
    )

    def power_fraction(lo, hi):
        mask = (
            (f >= lo)
            & (f < hi)
        )

        if mask.sum() == 0:
            return np.nan

        return float(
            np.sum(p[mask])
            / total
        )

    features[
        "bamboo_envelope_mod_3_6_fraction"
    ] = power_fraction(
        3.0,
        6.0,
    )

    features[
        "bamboo_envelope_mod_below4_fraction"
    ] = power_fraction(
        0.5,
        4.0,
    )

    probability = p / total
    positive_probability = probability[
        probability > 0
    ]

    if len(positive_probability) >= 2:
        entropy = float(
            -np.sum(
                positive_probability
                * np.log(
                    positive_probability
                )
            )
            / np.log(
                len(probability)
            )
        )

        features[
            "bamboo_envelope_mod_entropy"
        ] = float(
            np.clip(
                entropy,
                0.0,
                1.0,
            )
        )

    return features, diagnostics


# --------------------------------------------------------------
# Synthetic modulation validation
# --------------------------------------------------------------

sr_test = 16_000
t = (
    np.arange(
        int(8 * sr_test)
    )
    / sr_test
)

carrier = np.sin(
    2 * np.pi
    * 180.0
    * t
)

modulated = (
    1.0
    + 0.7
    * np.sin(
        2 * np.pi
        * 4.0
        * t
    )
) * carrier

env_test, _ = (
    envelope_rhythm_summary(
        modulated,
        sr_test,
        [(0.0, 8.0)],
    )
)

observed_peak = env_test[
    "bamboo_envelope_mod_peak_hz"
]

if (
    not np.isfinite(observed_peak)
    or abs(observed_peak - 4.0) > 0.5
):
    raise RuntimeError(
        "Envelope modulation validation failed: "
        f"expected ~4 Hz, observed {observed_peak}"
    )

print(
    "ENVELOPE-RHYTHM "
    "SYNTHETIC VALIDATION: PASS"
)


ENVELOPE-RHYTHM SYNTHETIC VALIDATION: PASS



## 5. Exact Goal 3 segmentation, perturbation, and Q extraction implementation

These are the same audited helper implementations used by Stage-B calibration.


In [10]:
ANALYSIS_SR = 16_000

SILERO_MODEL = None

def mono_native(native):
    values = np.asarray(native, dtype=np.float64)
    if values.ndim == 1:
        return values
    if values.ndim == 2:
        return values.mean(axis=1, dtype=np.float64)
    raise ValueError("Native audio must have shape samples or samples x channels.")

def native_to_analysis_16k(native, source_sr):
    mono = mono_native(native)
    mono = mono - float(np.mean(mono, dtype=np.float64))

    if int(source_sr) == ANALYSIS_SR:
        out = mono.astype(np.float32, copy=True)
    else:
        divisor = math.gcd(int(source_sr), ANALYSIS_SR)
        out = signal.resample_poly(
            mono,
            up=ANALYSIS_SR // divisor,
            down=int(source_sr) // divisor,
        ).astype(np.float32)

    if out.size == 0 or not np.isfinite(out).all():
        raise RuntimeError("Canonical analysis waveform is empty/non-finite.")

    return out

def canonical_resegmentation(analysis_16k):
    global SILERO_MODEL

    if SILERO_MODEL is None:
        SILERO_MODEL = load_silero_model(onnx=True)

    raw = silero_speech_intervals(
        np.asarray(analysis_16k, dtype=np.float32),
        threshold=0.5,
        min_speech_ms=250,
        min_silence_ms=100,
        speech_pad_ms=0,
        onnx=True,
        model=SILERO_MODEL,
    )

    return build_segmentation_views(
        raw,
        duration_sec=len(analysis_16k) / ANALYSIS_SR,
        bridge_gap_ms=100,
        min_speech_ms=250,
        strict_speech_edge_ms=50,
        strict_nonspeech_edge_ms=200,
    )

def intervals_as(items, cls, *, view=None):
    output = []

    for index, item in enumerate(items):
        start = float(item.start_sec)
        end = float(item.end_sec)

        if cls is QREVInterval:
            output.append(
                cls(
                    start,
                    end,
                    f"{view or 'segment'}_{index:05d}",
                    index,
                    view or "primary_speech",
                    "primary",
                )
            )
        else:
            output.append(cls(start, end))

    return output

def interval_support_sec(items):
    return float(
        sum(
            max(0.0, float(item.end_sec) - float(item.start_sec))
            for item in items
        )
    )

def task_span_from_primary(primary, duration_sec):
    if not primary:
        return None
    start = max(0.0, float(primary[0].start_sec))
    end = min(float(duration_sec), float(primary[-1].end_sec))
    return (start, end) if end > start else None

def strict_speech_ac_rms_native(native, source_sr, strict_intervals):
    mono = mono_native(native)
    pieces = []

    for item in strict_intervals:
        left = max(0, int(round(float(item.start_sec) * source_sr)))
        right = min(len(mono), int(round(float(item.end_sec) * source_sr)))
        if right > left:
            local = mono[left:right]
            local = local - float(np.mean(local))
            pieces.append(local)

    if not pieces:
        return np.nan

    values = np.concatenate(pieces)
    return float(np.sqrt(np.mean(values * values, dtype=np.float64)))

def colored_broadband_noise(n_samples, sample_rate_hz, seed):
    """Stationary 1/f-PSD broadband intervention field."""
    rng = np.random.default_rng(int(seed))
    frequencies = np.fft.rfftfreq(n_samples, d=1.0 / sample_rate_hz)

    spectrum = (
        rng.normal(size=len(frequencies))
        + 1j * rng.normal(size=len(frequencies))
    )

    amplitude = np.zeros_like(frequencies, dtype=np.float64)
    valid = frequencies >= 60.0
    amplitude[valid] = 1.0 / np.sqrt(frequencies[valid])

    shaped = np.fft.irfft(
        spectrum * amplitude,
        n=n_samples,
    ).astype(np.float64)

    shaped -= float(np.mean(shaped))
    scale = float(np.sqrt(np.mean(shaped * shaped)))

    if not np.isfinite(scale) or scale <= 0:
        raise RuntimeError("Could not synthesize colored interference.")

    return shaped / scale

def add_interference(
    native,
    source_sr,
    strict_speech_rms,
    *,
    snr_db,
    seed,
    amplitude_modulated,
):
    values = np.asarray(native, dtype=np.float64)
    if values.ndim == 1:
        values = values[:, None]

    if not np.isfinite(strict_speech_rms) or strict_speech_rms <= 0:
        raise RuntimeError("Baseline strict-speech RMS unavailable for injected SNR.")

    noise = colored_broadband_noise(
        values.shape[0],
        source_sr,
        seed,
    )

    if amplitude_modulated:
        time_axis = np.arange(len(noise), dtype=np.float64) / float(source_sr)

        # Smooth positive envelope. Re-normalize afterwards so candidate SNR
        # remains defined by total injected RMS.
        envelope = 0.35 + 0.65 * (
            0.5
            + 0.5 * np.sin(2.0 * np.pi * 0.35 * time_axis + 0.37)
        )

        noise = noise * envelope
        noise = noise / np.sqrt(np.mean(noise * noise))

    target_rms = strict_speech_rms / (
        10.0 ** (float(snr_db) / 20.0)
    )

    noise = noise * target_rms

    # Same acoustic field enters each channel. This is explicitly a
    # controlled intervention design, not a model of natural multichannel noise.
    return (values + noise[:, None]).astype(np.float32)

def apply_static_attenuation(native, gain_db):
    if float(gain_db) > 0:
        raise ValueError("Stage-B static gain grid must be attenuation-only.")
    return np.asarray(
        paper1_apply_gain_db(
            np.asarray(native, dtype=np.float64),
            float(gain_db),
        ),
        dtype=np.float32,
    )

DYNAMIC_GAIN_HZ = 0.70

def apply_dynamic_attenuation(native, source_sr, amplitude_db):
    """
    Smooth attenuation-only modulation in [-A, 0] dB.

    QGAIN's exact Paper 1 helper is applied channel by channel.
    """
    values = np.asarray(native, dtype=np.float64)
    if values.ndim == 1:
        values = values[:, None]

    amplitude_db = float(amplitude_db)
    if amplitude_db <= 0:
        raise ValueError("Dynamic gain amplitude must be positive.")

    time_axis = np.arange(values.shape[0], dtype=np.float64) / float(source_sr)

    envelope_db = (
        -amplitude_db / 2.0
        + amplitude_db / 2.0
        * np.sin(2.0 * np.pi * DYNAMIC_GAIN_HZ * time_axis)
    )

    channels = [
        apply_level_envelope_db(values[:, channel], envelope_db)
        for channel in range(values.shape[1])
    ]

    transformed = np.column_stack(channels)

    # No clipping competing mechanism: attenuation-only must never increase peak.
    baseline_peak = float(np.max(np.abs(values)))
    transformed_peak = float(np.max(np.abs(transformed)))

    if transformed_peak > baseline_peak * (1 + 1e-10) + 1e-12:
        raise RuntimeError("Dynamic attenuation unexpectedly increased waveform peak.")

    return transformed.astype(np.float32)

def paper1_preflight_synthetic_rir(rt60_sec, fs, seed):
    """
    Exact synthetic-RIR formula used by the pinned Paper 1 QREV analytical preflight.
    """
    rt60_sec = float(rt60_sec)
    if rt60_sec <= 0:
        return np.array([1.0], dtype=np.float64)

    rng = np.random.default_rng(int(seed))
    length = max(2, round((rt60_sec + 0.15) * fs))
    time_axis = np.arange(length, dtype=np.float64) / float(fs)

    tau = rt60_sec / np.log(1000.0)

    h = np.zeros(length, dtype=np.float64)
    h[0] = 1.0
    h += 0.08 * rng.standard_normal(length) * np.exp(-time_axis / tau)

    return h / np.sqrt(np.sum(h * h))

def apply_rir_rms_matched(native, source_sr, rt60_sec, seed):
    values = np.asarray(native, dtype=np.float64)
    if values.ndim == 1:
        values = values[:, None]

    h = paper1_preflight_synthetic_rir(
        rt60_sec,
        source_sr,
        seed,
    )

    channels = [
        signal.fftconvolve(values[:, channel], h)[: len(values)]
        for channel in range(values.shape[1])
    ]

    transformed = np.column_stack(channels)

    before = float(np.sqrt(np.mean(values * values)))
    after = float(np.sqrt(np.mean(transformed * transformed)))

    if not np.isfinite(before) or not np.isfinite(after) or after <= 0:
        raise RuntimeError("RIR RMS matching failed.")

    transformed *= before / after

    return transformed.astype(np.float32)

def apply_qchan_lowpass(native, source_sr, cutoff_hz, order):
    values = np.asarray(native, dtype=np.float64)
    if values.ndim == 1:
        values = values[:, None]

    channels = [
        paper1_qchan_lowpass_filter(
            values[:, channel],
            int(source_sr),
            float(cutoff_hz),
            order=int(order),
        )
        for channel in range(values.shape[1])
    ]

    return np.column_stack(channels).astype(np.float32)

print("CANONICAL SEGMENTATION + TRANSFORMATION HELPERS: READY")

CANONICAL SEGMENTATION + TRANSFORMATION HELPERS: READY


In [11]:
CORE_Q = [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
    "qrev_srmr_norm",
    "qchan_ltas_distance_db",
    "qchan_rolloff95_deficit_hz",
    "qchan_highband_ratio_deficit",
    "qchan_tilt_steepening_db_per_oct",
]

QDIST_TARGET = "qdist_hard_clipped_sample_fraction"

def blank_q_record():
    row = {feature: np.nan for feature in CORE_Q}
    row[QDIST_TARGET] = np.nan

    for feature in CORE_Q:
        row[f"{feature}__measurement_status"] = ""

    row[f"{QDIST_TARGET}__measurement_status"] = ""
    return row

def native_provenance_for_qdist(
    *,
    probe,
    source_path,
    source_sha256,
    variant_id,
):
    bits = probe.get("bits_per_raw_sample")

    try:
        bits = int(bits) if bits is not None and np.isfinite(float(bits)) else None
    except Exception:
        bits = None

    return qdist_detector.NativeSignalProvenance(
        native_view_verified=True,
        known_preprocessing_applied=False,
        codec_name=probe.get("codec_name"),
        sample_format=probe.get("sample_format"),
        bits_per_raw_sample=bits,
        container_format=probe.get("container_format"),
        channel_layout=probe.get("channel_layout"),
        source_path=f"goal3_controlled_in_memory::{variant_id}::{source_path}",
        source_sha256=source_sha256,
        decoded_sha256=None,
        decoder="ffmpeg controlled in-memory challenge",
        decoder_version="pinned source decode; transform documented separately",
        decode_arguments="first native decoded audio stream; channels preserved",
    )

def extract_all_q(
    native,
    source_sr,
    *,
    logical_recording_id,
    outer_fold,
    qchan_reference,
    qdist_requested,
    probe,
    source_path,
    source_sha256,
    variant_id,
):
    values = blank_q_record()
    errors = {}

    analysis = native_to_analysis_16k(native, source_sr)

    try:
        views = canonical_resegmentation(analysis)
        segmentation_status = "measured"
    except Exception as exc:
        return {
            **values,
            "segmentation_status": "error",
            "segmentation_error": f"{type(exc).__name__}: {exc}",
            "raw_speech_interval_count": 0,
            "primary_speech_interval_count": 0,
            "strict_speech_interval_count": 0,
            "strict_internal_nonspeech_interval_count": 0,
            "primary_speech_support_sec": 0.0,
            "strict_speech_support_sec": 0.0,
            "strict_internal_nonspeech_support_sec": 0.0,
            "analysis_duration_sec": len(analysis) / ANALYSIS_SR,
            "family_error_QADD": "",
            "family_error_QGAIN": "",
            "family_error_QREV": "",
            "family_error_QCHAN": "",
            "family_error_QDIST": "",
        }, None

    primary = views["primary_speech"]
    strict = views["strict_speech"]
    pauses = views["strict_internal_nonspeech"]

    # --------------------------------------------------------
    # QADD
    # --------------------------------------------------------
    try:
        qadd = extract_qadd(
            analysis,
            ANALYSIS_SR,
            primary_speech=intervals_as(primary, QADDInterval),
            strict_speech=intervals_as(strict, QADDInterval),
            strict_internal_nonspeech=intervals_as(pauses, QADDInterval),
            logical_recording_id=logical_recording_id,
            speech_intervals_are_guarded=True,
            pause_intervals_are_guarded=True,
        ).recording

        for feature in CORE_Q[:3]:
            values[feature] = qadd.get(feature, np.nan)
            values[f"{feature}__measurement_status"] = str(
                qadd.get(f"{feature}_status", "")
            )

    except Exception as exc:
        errors["QADD"] = f"{type(exc).__name__}: {exc}"

    # --------------------------------------------------------
    # QGAIN
    # --------------------------------------------------------
    try:
        qgain = extract_qgain(
            analysis,
            ANALYSIS_SR,
            strict_speech=intervals_as(strict, QGAINInterval),
            logical_recording_id=logical_recording_id,
            signal_provenance={
                "native_sample_rate_hz": int(source_sr),
                "native_channels": (
                    1
                    if np.asarray(native).ndim == 1
                    else int(np.asarray(native).shape[1])
                ),
                "codec_name": probe.get("codec_name"),
                "amplitude_normalization_applied": False,
                "denoising_applied": False,
                "dynamic_range_processing_applied": False,
            },
        ).recording

        for feature in CORE_Q[3:7]:
            values[feature] = qgain.get(feature, np.nan)
            values[f"{feature}__measurement_status"] = str(
                qgain.get(f"{feature}_status", "")
            )

    except Exception as exc:
        errors["QGAIN"] = f"{type(exc).__name__}: {exc}"

    # --------------------------------------------------------
    # QREV — primary Goal 3 verification uses normalized-fast SRMR
    # --------------------------------------------------------
    try:
        qrev = extract_qrev(
            analysis,
            ANALYSIS_SR,
            primary_speech=intervals_as(
                primary,
                QREVInterval,
                view="primary_speech",
            ),
            strict_speech=intervals_as(
                strict,
                QREVInterval,
                view="strict_speech",
            ),
            logical_recording_id=logical_recording_id,
            compute_srmr=True,
        ).recording

        feature = "qrev_srmr_norm"
        values[feature] = qrev.get(feature, np.nan)
        values[f"{feature}__measurement_status"] = str(
            qrev.get(f"{feature}_status", "")
        )

    except Exception as exc:
        errors["QREV"] = f"{type(exc).__name__}: {exc}"

    # --------------------------------------------------------
    # QCHAN — fixed unmodified outer-training reference
    # --------------------------------------------------------
    try:
        observation = extract_recording_spectrum(
            analysis,
            ANALYSIS_SR,
            strict_speech=intervals_as(strict, QCHANInterval),
            logical_recording_id=logical_recording_id,
            source_sample_rate_hz=int(source_sr),
        )

        qchan = compute_reference_relative_features(
            observation,
            qchan_reference,
        )

        for feature in CORE_Q[8:]:
            values[feature] = qchan.get(feature, np.nan)
            values[f"{feature}__measurement_status"] = str(
                qchan.get(f"{feature}_status", "")
            )

    except Exception as exc:
        errors["QCHAN"] = f"{type(exc).__name__}: {exc}"

    # --------------------------------------------------------
    # QDIST — native multichannel, only baseline/QDIST variants
    # --------------------------------------------------------
    if qdist_requested:
        try:
            span = task_span_from_primary(
                primary,
                len(analysis) / ANALYSIS_SR,
            )

            if span is None:
                raise RuntimeError(
                    "No primary-speech task span after re-segmentation."
                )

            qdist = qdist_detector.extract_qdist(
                np.asarray(native, dtype=np.float64),
                int(source_sr),
                task_span=qdist_detector.TimeInterval(*span),
                logical_recording_id=logical_recording_id,
                provenance=native_provenance_for_qdist(
                    probe=probe,
                    source_path=source_path,
                    source_sha256=source_sha256,
                    variant_id=variant_id,
                ),
            ).recording

            values[QDIST_TARGET] = qdist.get(QDIST_TARGET, np.nan)
            values[f"{QDIST_TARGET}__measurement_status"] = str(
                qdist.get("qdist_status", "")
            )

        except Exception as exc:
            errors["QDIST"] = f"{type(exc).__name__}: {exc}"

    row = {
        **values,
        "segmentation_status": segmentation_status,
        "segmentation_error": "",
        "raw_speech_interval_count": len(views["raw_speech"]),
        "primary_speech_interval_count": len(primary),
        "strict_speech_interval_count": len(strict),
        "strict_internal_nonspeech_interval_count": len(pauses),
        "primary_speech_support_sec": interval_support_sec(primary),
        "strict_speech_support_sec": interval_support_sec(strict),
        "strict_internal_nonspeech_support_sec": interval_support_sec(pauses),
        "analysis_duration_sec": len(analysis) / ANALYSIS_SR,
        "family_error_QADD": errors.get("QADD", ""),
        "family_error_QGAIN": errors.get("QGAIN", ""),
        "family_error_QREV": errors.get("QREV", ""),
        "family_error_QCHAN": errors.get("QCHAN", ""),
        "family_error_QDIST": errors.get("QDIST", ""),
    }

    return row, views

print("EXACT Q EXTRACTION ADAPTER: READY")

EXACT Q EXTRACTION ADAPTER: READY


In [12]:
def decode_source(source_row):
    media_path = Path(str(source_row["resolved_media_path"]))

    if not media_path.exists():
        raise FileNotFoundError(media_path)

    observed_hash = sha256_file(media_path)

    if observed_hash != str(source_row["observed_sha256"]):
        raise RuntimeError(
            f"Source hash changed since Stage-B preparation: {media_path}"
        )

    decoded = decode_native_audio(
        media_path,
        ffmpeg=shutil.which("ffmpeg") or "ffmpeg",
        ffprobe=shutil.which("ffprobe") or "ffprobe",
    )

    native = np.asarray(decoded.native, dtype=np.float32)
    source_sr = int(decoded.sample_rate_native)

    analysis = native_to_analysis_16k(native, source_sr)
    views = canonical_resegmentation(analysis)

    strict_rms = strict_speech_ac_rms_native(
        native,
        source_sr,
        views["strict_speech"],
    )

    span = task_span_from_primary(
        views["primary_speech"],
        len(analysis) / ANALYSIS_SR,
    )

    if span is None:
        raise RuntimeError(
            "Baseline re-segmentation yielded no primary task span."
        )

    return {
        "native": native,
        "source_sr": source_sr,
        "probe": decoded.probe,
        "baseline_views": views,
        "baseline_strict_speech_rms_native": strict_rms,
        "baseline_task_span_sec": span,
        "media_path": media_path,
        "source_sha256": observed_hash,
    }

def transform_candidate(source, candidate, source_row):
    native = source["native"]
    source_sr = source["source_sr"]
    transform = str(candidate["transform"])
    value = float(candidate["candidate_value"])
    exemplar = int(candidate["exemplar"])

    metadata = {
        "random_seed": np.nan,
        "filter_order": candidate.get("filter_order", np.nan),
        "rir_seed": candidate.get("rir_seed", np.nan),
        "qdist_target_fraction": np.nan,
        "qdist_realized_fraction": np.nan,
        "qdist_positive_limit": np.nan,
        "qdist_negative_limit": np.nan,
        "dynamic_gain_frequency_hz": np.nan,
    }

    if transform == "stationary_colored_broadband":
        seed = stable_seed(
            "QADD_stationary",
            source_row["logical_recording_id"],
            value,
            exemplar,
        )
        transformed = add_interference(
            native,
            source_sr,
            source["baseline_strict_speech_rms_native"],
            snr_db=value,
            seed=seed,
            amplitude_modulated=False,
        )
        metadata["random_seed"] = int(seed)

    elif transform == "amplitude_modulated_colored_broadband":
        seed = stable_seed(
            "QADD_amplitude_modulated",
            source_row["logical_recording_id"],
            value,
            exemplar,
        )
        transformed = add_interference(
            native,
            source_sr,
            source["baseline_strict_speech_rms_native"],
            snr_db=value,
            seed=seed,
            amplitude_modulated=True,
        )
        metadata["random_seed"] = int(seed)

    elif transform == "uniform_level_shift":
        transformed = apply_static_attenuation(native, value)

    elif transform == "smooth_time_varying_gain":
        transformed = apply_dynamic_attenuation(
            native,
            source_sr,
            value,
        )
        metadata["dynamic_gain_frequency_hz"] = DYNAMIC_GAIN_HZ

    elif transform == "RIR_convolution_RMS_matched":
        rir_seed = int(candidate["rir_seed"])
        transformed = apply_rir_rms_matched(
            native,
            source_sr,
            value,
            rir_seed,
        )
        metadata["rir_seed"] = rir_seed

    elif transform == "upper_band_restriction":
        filter_order = int(candidate["filter_order"])
        transformed = apply_qchan_lowpass(
            native,
            source_sr,
            value,
            filter_order,
        )
        metadata["filter_order"] = filter_order

    elif transform == "symmetric_hard_clipping":
        start_sec, end_sec = source["baseline_task_span_sec"]
        left = max(0, int(math.floor(start_sec * source_sr)))
        right = min(
            native.shape[0],
            int(math.ceil(end_sec * source_sr)),
        )

        if right <= left:
            raise RuntimeError("Invalid baseline task span for QDIST intervention.")

        crop = np.asarray(native[left:right], dtype=np.float64)

        altered_crop, truth, limits = inject_matched_hard_clip(
            crop,
            value,
            QDIST_GEOMETRY,
        )

        transformed = np.asarray(native, dtype=np.float64).copy()
        transformed[left:right] = altered_crop
        transformed = transformed.astype(np.float32)

        metadata.update({
            "qdist_target_fraction": value,
            "qdist_realized_fraction": float(truth.mean()),
            "qdist_positive_limit": limits.get("positive_limit", np.nan),
            "qdist_negative_limit": limits.get("negative_limit", np.nan),
        })

    else:
        raise ValueError(f"Unknown transform: {transform}")

    if transformed.shape != np.asarray(native).shape:
        raise RuntimeError(
            f"Transformation changed waveform shape: {transform}"
        )

    if not np.isfinite(transformed).all():
        raise RuntimeError(
            f"Transformation produced non-finite samples: {transform}"
        )

    return transformed, metadata

print("SOURCE + CANDIDATE TRANSFORMATION ENGINE: READY")

SOURCE + CANDIDATE TRANSFORMATION ENGINE: READY



## 6. A-extraction adapter for freshly re-segmented waveforms

This wrapper uses the frozen v2.1 feature functions but substitutes the current waveform's newly measured `primary_speech` intervals.  
Missingness/support status is retained as an outcome.


In [13]:

def intervals_to_tuples(items):
    return [
        (float(item.start_sec), float(item.end_sec))
        for item in items
        if float(item.end_sec) > float(item.start_sec)
    ]


def blank_a_row():
    row = {feature: np.nan for feature in FEATURES}
    row.update({column: 0 for column in SUPPORT_COLUMNS})
    return row


def finalize_a_support(row):
    for feature in FEATURES:
        value = pd.to_numeric(
            pd.Series([row.get(feature)]),
            errors="coerce",
        ).iloc[0]

        row[f"{feature}__supported"] = int(
            np.isfinite(value)
        )

    return row


def extract_a_from_primary_intervals(
    analysis_16k,
    primary,
):
    """
    Exact v2.1 acoustic measurement implementation given an explicit
    primary-speech interval list.

    This function deliberately does not decide where the intervals came from.
    It can therefore be used for:

      1. the exact frozen-A reproduction path using the frozen/adjudicated
         primary_speech intervals; and
      2. the Goal-3 end-to-end path using freshly re-segmented intervals.

    Keeping those two paths separate is essential because the frozen
    segmentation corpus can contain reviewed/manual boundary corrections,
    whereas Goal 3 primary perturbation analysis intentionally re-runs Silero.
    """

    analysis = np.asarray(
        analysis_16k,
        dtype=float,
    )

    if (
        analysis.ndim != 1
        or analysis.size == 0
        or not np.isfinite(analysis).all()
    ):
        raise RuntimeError(
            "A adapter received invalid 16-kHz mono waveform."
        )

    primary = [
        (float(start), float(end))
        for start, end in primary
        if (
            np.isfinite(start)
            and np.isfinite(end)
            and float(end) > float(start)
        )
    ]

    row = {
        "acoustic_extraction_status": "failed",
        "acoustic_extraction_error": "",
        "timing_status": "not_run",
        "pitch_status": "not_run",
        "formant_status": "not_run",
        "envelope_status": "not_run",
        "analysis_sample_rate_hz": TARGET_SR,
        "primary_interval_count": len(primary),
        "pitch_frames_analyzed": 0,
        "pitch_frames_voiced": 0,
        "formant_frames_analyzed": 0,
        "formant_frames_valid": 0,
        "formant_valid_frame_fraction": np.nan,
        "envelope_support_sec": np.nan,
    }

    row.update(
        blank_a_row()
    )

    if not primary:
        row["acoustic_extraction_error"] = (
            "no_primary_speech_intervals"
        )
        return finalize_a_support(row)

    errors = []

    try:
        row.update(
            timing_features_for(primary)
        )
        row["timing_status"] = "ok"

    except Exception as exc:
        row["timing_status"] = "failed"
        errors.append(
            f"timing:{type(exc).__name__}:{exc}"
        )

    try:
        pitch, diagnostics = pitch_summary(
            analysis,
            TARGET_SR,
            primary,
        )

        row.update(pitch)
        row.update(diagnostics)

        row["pitch_status"] = (
            "ok"
            if np.isfinite(
                row["bamboo_f0_median_hz"]
            )
            else "insufficient_support"
        )

    except Exception as exc:
        row["pitch_status"] = "failed"
        errors.append(
            f"pitch:{type(exc).__name__}:{exc}"
        )

    try:
        formants, diagnostics = (
            formant_summary(
                analysis,
                TARGET_SR,
                primary,
            )
        )

        row.update(formants)
        row.update(diagnostics)

        row["formant_status"] = (
            "ok"
            if np.isfinite(
                row["bamboo_f2_iqr_hz"]
            )
            else "insufficient_support"
        )

    except Exception as exc:
        row["formant_status"] = "failed"
        errors.append(
            f"formant:{type(exc).__name__}:{exc}"
        )

    try:
        envelope, diagnostics = (
            envelope_rhythm_summary(
                analysis,
                TARGET_SR,
                primary,
            )
        )

        row.update(envelope)
        row.update(diagnostics)

        row["envelope_status"] = (
            "ok"
            if np.isfinite(
                row["bamboo_envelope_mod_peak_hz"]
            )
            else "insufficient_support"
        )

    except Exception as exc:
        row["envelope_status"] = "failed"
        errors.append(
            f"envelope:{type(exc).__name__}:{exc}"
        )

    row = finalize_a_support(row)

    n_available = int(
        sum(
            row[column]
            for column in SUPPORT_COLUMNS
        )
    )

    if n_available == len(FEATURES):
        row["acoustic_extraction_status"] = (
            "complete"
        )
    elif n_available > 0:
        row["acoustic_extraction_status"] = (
            "partial"
        )
    else:
        row["acoustic_extraction_status"] = (
            "failed"
        )

    row["acoustic_extraction_error"] = (
        " | ".join(errors)
    )

    return row


def extract_a_from_analysis_views(
    analysis_16k,
    views,
):
    """
    Goal-3 primary end-to-end A extraction from freshly measured views.
    """

    primary = intervals_to_tuples(
        views["primary_speech"]
    )

    return extract_a_from_primary_intervals(
        analysis_16k,
        primary,
    )


print(
    "A ADAPTER WITH FROZEN/FRESH SEGMENTATION SEPARATION: READY"
)


A ADAPTER WITH FROZEN/FRESH SEGMENTATION SEPARATION: READY



## 7. Two-path baseline verification

The frozen v2.1 acoustic representation was extracted from the project's **frozen/adjudicated `primary_speech` intervals**. Those intervals may include review/manual corrections for atypical recordings and therefore are not required to be exactly reproduced by a new Silero pass.

This preflight separates two different questions:

1. **Hard implementation reproduction gate.** Using the exact frozen/adjudicated intervals, the v2.1 acoustic code reproduced here must numerically reproduce frozen A and its support indicators.
2. **Fresh re-segmentation audit.** The unmodified waveform is independently re-segmented through the Goal 3 primary path. Differences from the frozen/adjudicated intervals are recorded as segmentation-mediated measurement differences rather than treated as software errors.

This matches the Goal 3 analysis specification: the **primary** controlled perturbation analysis reruns segmentation, while fixed-baseline segmentation is a prespecified sensitivity.


In [14]:

FROZEN_SEGMENTS_PATH = (
    RAW
    / "segments"
    / "frozen_segmentation_intervals.csv"
)

frozen_segments = read_csv_checked(
    FROZEN_SEGMENTS_PATH,
    required=[
        "logical_recording_id",
        "start_sec",
        "end_sec",
        "view",
    ],
)

frozen_segments["logical_recording_id"] = (
    frozen_segments[
        "logical_recording_id"
    ].astype(str)
)

frozen_segments["start_sec"] = (
    pd.to_numeric(
        frozen_segments["start_sec"],
        errors="coerce",
    )
)

frozen_segments["end_sec"] = (
    pd.to_numeric(
        frozen_segments["end_sec"],
        errors="coerce",
    )
)

frozen_segments["view"] = (
    frozen_segments["view"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"[\s\-]+",
        "_",
        regex=True,
    )
)

primary_intervals = (
    frozen_segments.loc[
        frozen_segments[
            "view"
        ].eq("primary_speech")
    ]
    .copy()
)

frozen_values_lookup = (
    a_values
    .assign(
        logical_recording_id=lambda d: (
            d["logical_recording_id"]
            .astype(str)
        )
    )
    .set_index(
        "logical_recording_id"
    )
)

registry_by_feature = (
    a_registry.set_index(
        "feature"
    )
)

A_RTOL = 1e-6
A_ATOL = 1e-8

fixed_reproduction_rows = []
fresh_divergence_rows = []
baseline_cache = {}
frozen_primary_by_recording = {}


def _finite_or_nan(value):
    return pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]


for source_row in (
    preflight_sources.to_dict(
        "records"
    )
):
    rid = str(
        source_row[
            "logical_recording_id"
        ]
    )

    source = decode_source(
        source_row
    )

    baseline_cache[rid] = source

    analysis = (
        native_to_analysis_16k(
            source["native"],
            source["source_sr"],
        )
    )

    # --------------------------------------------------------------
    # A. EXACT MODEL-COMPATIBLE FROZEN-SEGMENTATION PATH
    #
    # Notebook 18 extracted frozen A from the previously adjudicated
    # primary_speech table.  Some retained recordings can therefore
    # contain reviewed/manual boundary corrections rather than a
    # fresh Silero rerun.
    #
    # THIS is the path that must numerically reproduce frozen A.
    # --------------------------------------------------------------

    old_rows_raw = (
        primary_intervals.loc[
            primary_intervals[
                "logical_recording_id"
            ].eq(rid),
            ["start_sec", "end_sec"],
        ]
        .sort_values(
            ["start_sec", "end_sec"]
        )
    )

    frozen_primary = (
        merge_overlapping_intervals(
            old_rows_raw
        )
    )

    frozen_primary_by_recording[
        rid
    ] = frozen_primary

    if not frozen_primary:
        raise RuntimeError(
            f"{rid}: no frozen primary_speech intervals."
        )

    fixed_measured = (
        extract_a_from_primary_intervals(
            analysis,
            frozen_primary,
        )
    )

    frozen = (
        frozen_values_lookup.loc[
            rid
        ]
    )

    for feature in RETAINED_A:

        support_col = str(
            registry_by_feature.loc[
                feature,
                "support_indicator_column",
            ]
        )

        frozen_value = (
            _finite_or_nan(
                frozen[feature]
            )
        )

        measured_value = (
            _finite_or_nan(
                fixed_measured.get(
                    feature
                )
            )
        )

        frozen_supported = int(
            pd.to_numeric(
                pd.Series(
                    [
                        frozen[
                            support_col
                        ]
                    ]
                ),
                errors="raise",
            ).iloc[0]
        )

        measured_supported = int(
            fixed_measured.get(
                support_col,
                0,
            )
        )

        support_match = (
            frozen_supported
            == measured_supported
        )

        missingness_match = (
            np.isfinite(
                frozen_value
            )
            == np.isfinite(
                measured_value
            )
        )

        if (
            np.isfinite(
                frozen_value
            )
            and np.isfinite(
                measured_value
            )
        ):

            numerical_match = bool(
                np.isclose(
                    frozen_value,
                    measured_value,
                    rtol=A_RTOL,
                    atol=A_ATOL,
                )
            )

            abs_diff = float(
                abs(
                    frozen_value
                    - measured_value
                )
            )

        else:
            numerical_match = bool(
                missingness_match
            )
            abs_diff = np.nan

        fixed_reproduction_rows.append(
            {
                "logical_recording_id": (
                    rid
                ),
                "feature": feature,
                "final_role": (
                    registry_by_feature.loc[
                        feature,
                        "final_role",
                    ]
                ),
                "frozen_value": (
                    frozen_value
                ),
                "remeasured_fixed_segmentation_value": (
                    measured_value
                ),
                "absolute_difference": (
                    abs_diff
                ),
                "frozen_supported": (
                    frozen_supported
                ),
                "remeasured_supported": (
                    measured_supported
                ),
                "support_match": (
                    support_match
                ),
                "missingness_match": (
                    missingness_match
                ),
                "numerical_match": (
                    numerical_match
                ),
                "raw_frozen_primary_rows": int(
                    len(old_rows_raw)
                ),
                "merged_frozen_primary_intervals": int(
                    len(frozen_primary)
                ),
            }
        )

    # --------------------------------------------------------------
    # B. GOAL-3 PRIMARY FRESH RE-SEGMENTATION PATH
    #
    # The Methods explicitly require the primary perturbation path
    # to rerun segmentation.  Fresh baseline segmentation therefore
    # need NOT equal the adjudicated frozen interval table.
    #
    # Any difference here is retained as a segmentation-mediated
    # measurement shift, not treated as a software failure.
    # --------------------------------------------------------------

    fresh_primary = (
        intervals_to_tuples(
            source[
                "baseline_views"
            ][
                "primary_speech"
            ]
        )
    )

    fresh_measured = (
        extract_a_from_primary_intervals(
            analysis,
            fresh_primary,
        )
    )

    frozen_support_sec = float(
        np.sum(
            [
                end - start
                for start, end
                in frozen_primary
            ]
        )
    )

    fresh_support_sec = float(
        np.sum(
            [
                end - start
                for start, end
                in fresh_primary
            ]
        )
    )

    fresh_divergence_rows.append(
        {
            "logical_recording_id": (
                rid
            ),
            "raw_frozen_primary_rows": int(
                len(old_rows_raw)
            ),
            "merged_frozen_primary_intervals": int(
                len(frozen_primary)
            ),
            "fresh_primary_intervals": int(
                len(fresh_primary)
            ),
            "frozen_primary_support_sec": (
                frozen_support_sec
            ),
            "fresh_primary_support_sec": (
                fresh_support_sec
            ),
            "fresh_minus_frozen_support_sec": (
                fresh_support_sec
                - frozen_support_sec
            ),
            "fresh_extraction_status": (
                fresh_measured[
                    "acoustic_extraction_status"
                ]
            ),
            "fresh_extraction_error": (
                fresh_measured[
                    "acoustic_extraction_error"
                ]
            ),
        }
    )


fixed_reproduction = pd.DataFrame(
    fixed_reproduction_rows
)

fresh_segmentation_audit = (
    pd.DataFrame(
        fresh_divergence_rows
    )
)


atomic_csv(
    fixed_reproduction,
    TABLES
    / (
        "baseline_A_reproduction_"
        "fixed_frozen_segmentation.csv"
    ),
)

atomic_csv(
    fresh_segmentation_audit,
    TABLES
    / (
        "baseline_fresh_resegmentation_"
        "divergence_audit.csv"
    ),
)


fixed_failures = (
    fixed_reproduction.loc[
        ~(
            fixed_reproduction[
                "support_match"
            ].astype(bool)
            & fixed_reproduction[
                "missingness_match"
            ].astype(bool)
            & fixed_reproduction[
                "numerical_match"
            ].astype(bool)
        )
    ]
    .copy()
)


print(
    "EXACT FROZEN-A REPRODUCTION "
    "USING FROZEN/ADJUDICATED SEGMENTATION"
)

print(
    "Comparisons:",
    len(fixed_reproduction),
)

print(
    "Failures:",
    len(fixed_failures),
)


if len(fixed_failures):

    display(
        fixed_failures
    )

    raise RuntimeError(
        "FROZEN-A IMPLEMENTATION REPRODUCTION FAILED "
        "EVEN WHEN USING THE SAME FROZEN SEGMENTATION. "
        "Do not proceed."
    )


print()
print(
    "FROZEN-A IMPLEMENTATION REPRODUCTION: PASS"
)


display(
    fixed_reproduction.groupby(
        [
            "logical_recording_id",
            "final_role",
        ],
        as_index=False,
    ).agg(
        features=(
            "feature",
            "size",
        ),
        max_abs_difference=(
            "absolute_difference",
            "max",
        ),
        frozen_primary_intervals=(
            "merged_frozen_primary_intervals",
            "max",
        ),
    )
)


print()
print(
    "FRESH RE-SEGMENTATION DIVERGENCE AUDIT "
    "(EXPECTED TO BE NONZERO FOR SOME ADJUDICATED RECORDINGS)"
)

display(
    fresh_segmentation_audit
)


# Compatibility aliases used by the final seal cell.
reproduction = fixed_reproduction
failed_reproduction = fixed_failures


print()
print(
    "MEASUREMENT-PATH GATE: PASS"
)

print(
    "Interpretation:"
)

print(
    "  • Frozen-A code reproduction under its actual "
    "frozen segmentation = PASS."
)

print(
    "  • Fresh Goal-3 baseline segmentation is audited "
    "separately and is not required to equal adjudicated "
    "frozen segmentation."
)

print(
    "  • Primary Goal-3 perturbation analysis will rerun "
    "segmentation for baseline and perturbed waveforms."
)

print(
    "  • Fixed-frozen-segmentation analysis remains a "
    "prespecified sensitivity."
)


EXACT FROZEN-A REPRODUCTION USING FROZEN/ADJUDICATED SEGMENTATION
Comparisons: 56
Failures: 0

FROZEN-A IMPLEMENTATION REPRODUCTION: PASS


,logical_recording_id,final_role,features,max_abs_difference,frozen_primary_intervals
0,CAPT0000099_272_1_20240523_240_PSG_BAMBOO,extended,22,1.010889e-04,9
1,CAPT0000099_272_1_20240523_240_PSG_BAMBOO,primary,6,3.101265e-08,9
2,CAPT0000114_272_1_20240806_240_PSG_BAMBOO,extended,22,1.666238e-05,45
3,CAPT0000114_272_1_20240806_240_PSG_BAMBOO,primary,6,2.364320e-08,45



FRESH RE-SEGMENTATION DIVERGENCE AUDIT (EXPECTED TO BE NONZERO FOR SOME ADJUDICATED RECORDINGS)


,logical_recording_id,raw_frozen_primary_rows,merged_frozen_primary_intervals,fresh_primary_intervals,frozen_primary_support_sec,fresh_primary_support_sec,fresh_minus_frozen_support_sec,fresh_extraction_status,fresh_extraction_error
0,CAPT0000114_272_1_20240806_240_PSG_BAMBOO,132,45,47,68.192,56.608,-11.584,complete,
1,CAPT0000099_272_1_20240523_240_PSG_BAMBOO,29,9,10,26.080,25.664,-0.416,complete,



MEASUREMENT-PATH GATE: PASS
Interpretation:
  • Frozen-A code reproduction under its actual frozen segmentation = PASS.
  • Fresh Goal-3 baseline segmentation is audited separately and is not required to equal adjudicated frozen segmentation.
  • Primary Goal-3 perturbation analysis will rerun segmentation for baseline and perturbed waveforms.
  • Fixed-frozen-segmentation analysis remains a prespecified sensitivity.



## 8. Tiny controlled Q+A execution smoke test

The sealed dose manifest is not modified.  
For the two pilot sources, this smoke test expands:

- low and high dose only;
- exemplar 1 for stochastic/exemplar transforms;
- exact QREV seed `440`;
- exact QCHAN Butterworth order `4`;
- exact deterministic stationary-QADD realization rule;
- exact QDIST symmetric clipping rule.

The purpose is to exercise all six final transform paths plus Q and A extraction.  
Dose-response inference is **not** performed on this two-source smoke test.


In [15]:

# Build a compact execution grid from the sealed dose manifest.
smoke_doses = dose_manifest.loc[
    dose_manifest["dose_label"].isin(["low", "high"])
].copy()

execution_rows = []

for row in smoke_doses.to_dict("records"):
    transform = str(row["transform"])

    execution_row = {
        "family": str(row["family"]),
        "transform": transform,
        "dose_label": str(row["dose_label"]),
        "candidate_value": float(row["candidate_value"]),
        "candidate_unit": str(row["candidate_unit"]),
        "physical_strength_order": float(
            row.get("physical_strength_order", np.nan)
        ),
        "exemplar": 1,
        "filter_order": np.nan,
        "rir_seed": np.nan,
    }

    if transform == "RIR_convolution_RMS_matched":
        execution_row["rir_seed"] = 440

    if transform == "upper_band_restriction":
        execution_row["filter_order"] = 4

    execution_row["candidate_id"] = stable_hash(
        ENGINE_VERSION,
        execution_row["family"],
        execution_row["transform"],
        execution_row["dose_label"],
        execution_row["candidate_value"],
        execution_row["exemplar"],
        execution_row["filter_order"],
        execution_row["rir_seed"],
    )[:20]

    execution_rows.append(execution_row)

execution_grid = pd.DataFrame(execution_rows)

if len(execution_grid) != 12:
    raise RuntimeError(
        f"Expected 12 smoke perturbation rows, found {len(execution_grid)}."
    )

display(
    execution_grid[
        [
            "family",
            "transform",
            "dose_label",
            "candidate_value",
            "candidate_unit",
            "exemplar",
            "filter_order",
            "rir_seed",
        ]
    ]
)

print("SEALED-GRID SMOKE EXPANSION: PASS")


,family,transform,dose_label,candidate_value,candidate_unit,exemplar,filter_order,rir_seed
0,QADD,stationary_colored_broadband,low,35.000,dB injected SNR,1,NaN,NaN
1,QADD,stationary_colored_broadband,high,10.000,dB injected SNR,1,NaN,NaN
2,QCHAN,upper_band_restriction,low,4500.000,low-pass cutoff Hz,1,4.0,NaN
3,QCHAN,upper_band_restriction,high,2000.000,low-pass cutoff Hz,1,4.0,NaN
4,QDIST,symmetric_hard_clipping,low,0.001,target changed channel-sample fraction,1,NaN,NaN
5,QDIST,symmetric_hard_clipping,high,0.010,target changed channel-sample fraction,1,NaN,NaN
6,QGAIN,smooth_time_varying_gain,low,10.000,dB modulation amplitude,1,NaN,NaN
7,QGAIN,smooth_time_varying_gain,high,24.000,dB modulation amplitude,1,NaN,NaN
8,QGAIN,uniform_level_shift,low,-3.000,dB gain,1,NaN,NaN
9,QGAIN,uniform_level_shift,high,-10.000,dB gain,1,NaN,NaN


SEALED-GRID SMOKE EXPANSION: PASS


In [16]:

def preflight_checkpoint_path(source_id, variant_id):
    folder = CHECKPOINTS / safe_slug(source_id)
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{variant_id}.json"

def preflight_checkpoint_signature(source_row, contract):
    return stable_hash(
        ENGINE_VERSION,
        PAPER1_COMMIT,
        manifest_hash,
        a_freeze_manifest["acoustic_features_frozen_sha256"],
        source_row["logical_recording_id"],
        source_row["observed_sha256"],
        json.dumps(json_safe(contract), sort_keys=True),
    )

def load_preflight_checkpoint(path, signature):
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None
    if payload.get("signature") != signature:
        return None
    if payload.get("engine_version") != ENGINE_VERSION:
        return None
    return payload.get("row")

def save_preflight_checkpoint(path, signature, row):
    atomic_json(
        {
            "created_utc": utc_now(),
            "engine_version": ENGINE_VERSION,
            "signature": signature,
            "row": row,
        },
        path,
    )

all_rows = []

for source_row in preflight_sources.to_dict("records"):
    rid = str(source_row["logical_recording_id"])
    pid = str(source_row["participant_id"])
    fold = int(source_row["outer_fold"])

    source = baseline_cache.get(rid)
    if source is None:
        source = decode_source(source_row)

    qchan_reference = fixed_qchan_references[fold]

    variants = [
        {
            "family": "BASELINE",
            "transform": "baseline",
            "dose_label": "baseline",
            "candidate_value": 0.0,
            "candidate_unit": "none",
            "physical_strength_order": 0.0,
            "exemplar": 0,
            "filter_order": np.nan,
            "rir_seed": np.nan,
            "candidate_id": "baseline",
        }
    ] + execution_grid.to_dict("records")

    for candidate in variants:
        variant_id = stable_hash(
            ENGINE_VERSION,
            rid,
            candidate["candidate_id"],
        )[:20]

        contract = {
            **candidate,
            "variant_id": variant_id,
        }

        signature = preflight_checkpoint_signature(
            source_row,
            contract,
        )

        cp = preflight_checkpoint_path(rid, variant_id)
        cached = load_preflight_checkpoint(cp, signature)

        if cached is not None:
            all_rows.append(cached)
            continue

        started = time.perf_counter()

        try:
            if candidate["transform"] == "baseline":
                waveform = source["native"]
                transform_meta = {
                    "random_seed": np.nan,
                    "filter_order": np.nan,
                    "rir_seed": np.nan,
                    "qdist_target_fraction": np.nan,
                    "qdist_realized_fraction": np.nan,
                    "qdist_positive_limit": np.nan,
                    "qdist_negative_limit": np.nan,
                    "dynamic_gain_frequency_hz": np.nan,
                }
            else:
                waveform, transform_meta = transform_candidate(
                    source,
                    candidate,
                    source_row,
                )

            q_values, views = extract_all_q(
                waveform,
                source["source_sr"],
                logical_recording_id=f"{rid}__qa_preflight__{variant_id}",
                outer_fold=fold,
                qchan_reference=qchan_reference,
                qdist_requested=(
                    candidate["family"] == "QDIST"
                    or candidate["transform"] == "baseline"
                ),
                probe=source["probe"],
                source_path=str(source["media_path"]),
                source_sha256=source["source_sha256"],
                variant_id=variant_id,
            )

            if views is None:
                raise RuntimeError("Segmentation failed; no views returned.")

            analysis = native_to_analysis_16k(
                waveform,
                source["source_sr"],
            )

            a_measured = extract_a_from_analysis_views(
                analysis,
                views,
            )

            row = {
                "participant_id": pid,
                "logical_recording_id": rid,
                "outer_fold": fold,
                "pilot_rank": int(source_row["pilot_rank"]),
                "variant_id": variant_id,
                "family": candidate["family"],
                "transform": candidate["transform"],
                "dose_label": candidate["dose_label"],
                "candidate_value": float(candidate["candidate_value"]),
                "candidate_unit": candidate["candidate_unit"],
                "exemplar": int(candidate["exemplar"]),
                "source_sha256": source["source_sha256"],
                "execution_status": "PASS",
                "execution_error": "",
                "elapsed_sec": time.perf_counter() - started,
                **transform_meta,
                **q_values,
                **{
                    f"A__{feature}": a_measured.get(feature, np.nan)
                    for feature in RETAINED_A
                },
                **{
                    f"A_support__{feature}": a_measured.get(
                        str(
                            a_registry.set_index("feature").loc[
                                feature,
                                "support_indicator_column",
                            ]
                        ),
                        0,
                    )
                    for feature in RETAINED_A
                },
                "A_extraction_status": a_measured[
                    "acoustic_extraction_status"
                ],
                "A_extraction_error": a_measured[
                    "acoustic_extraction_error"
                ],
            }

        except Exception as exc:
            row = {
                "participant_id": pid,
                "logical_recording_id": rid,
                "outer_fold": fold,
                "pilot_rank": int(source_row["pilot_rank"]),
                "variant_id": variant_id,
                "family": candidate["family"],
                "transform": candidate["transform"],
                "dose_label": candidate["dose_label"],
                "candidate_value": float(candidate["candidate_value"]),
                "candidate_unit": candidate["candidate_unit"],
                "exemplar": int(candidate["exemplar"]),
                "source_sha256": source["source_sha256"],
                "execution_status": "ERROR",
                "execution_error": f"{type(exc).__name__}: {exc}",
                "elapsed_sec": time.perf_counter() - started,
            }

        save_preflight_checkpoint(cp, signature, row)
        all_rows.append(row)

response = pd.DataFrame(all_rows)

atomic_csv(
    response,
    TABLES / "controlled_QA_preflight_response.csv",
)

expected_rows = EXPECTED["preflight_sources"] * (1 + len(execution_grid))

if len(response) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} response rows, found {len(response)}."
    )

errors = response.loc[
    ~response["execution_status"].eq("PASS")
].copy()

print("Controlled Q+A preflight rows:", len(response))
print("Execution errors:", len(errors))

if len(errors):
    display(
        errors[
            [
                "logical_recording_id",
                "transform",
                "dose_label",
                "execution_error",
            ]
        ]
    )
    raise RuntimeError(
        "CONTROLLED Q+A EXECUTION PREFLIGHT FAILED."
    )

print("CONTROLLED Q+A EXECUTION: PASS")


Controlled Q+A preflight rows: 26
Execution errors: 0
CONTROLLED Q+A EXECUTION: PASS



## 9. Paired ΔQ and ΔA output

Support loss and feature unavailability are retained.  
No significance test or clinical interpretation is performed here.


In [17]:

baseline = response.loc[
    response["transform"].eq("baseline")
].copy()

if baseline["logical_recording_id"].nunique() != EXPECTED["preflight_sources"]:
    raise RuntimeError("Baseline response is incomplete.")

delta = response.loc[
    ~response["transform"].eq("baseline")
].copy()

q_delta_features = CORE_Q + [QDIST_TARGET]

baseline_columns = (
    ["logical_recording_id"]
    + q_delta_features
    + [f"A__{feature}" for feature in RETAINED_A]
)

baseline_lookup = baseline[baseline_columns].copy()

baseline_lookup = baseline_lookup.rename(
    columns={
        column: f"baseline__{column}"
        for column in baseline_columns
        if column != "logical_recording_id"
    }
)

delta = delta.merge(
    baseline_lookup,
    on="logical_recording_id",
    how="left",
    validate="many_to_one",
)

for feature in q_delta_features:
    delta[f"delta_Q__{feature}"] = (
        pd.to_numeric(delta.get(feature), errors="coerce")
        - pd.to_numeric(
            delta.get(f"baseline__{feature}"),
            errors="coerce",
        )
    )

for feature in RETAINED_A:
    current = f"A__{feature}"
    baseline_col = f"baseline__A__{feature}"
    delta[f"delta_A__{feature}"] = (
        pd.to_numeric(delta.get(current), errors="coerce")
        - pd.to_numeric(delta.get(baseline_col), errors="coerce")
    )

atomic_csv(
    delta,
    TABLES / "controlled_QA_preflight_delta.csv",
)

summary = (
    delta.groupby(
        ["family", "transform", "dose_label"],
        as_index=False,
    )
    .agg(
        rows=("variant_id", "size"),
        sources=("logical_recording_id", "nunique"),
        mean_elapsed_sec=("elapsed_sec", "mean"),
    )
)

display(summary)

print("PAIRED ΔQ / ΔA TABLE: WRITTEN")
print(
    "No clinical outcome, prediction, loss, or model object was loaded."
)


,family,transform,dose_label,rows,sources,mean_elapsed_sec
0,QADD,stationary_colored_broadband,high,2,2,1.390915
1,QADD,stationary_colored_broadband,low,2,2,1.955119
2,QCHAN,upper_band_restriction,high,2,2,1.125475
3,QCHAN,upper_band_restriction,low,2,2,1.634403
4,QDIST,symmetric_hard_clipping,high,2,2,2.125282
5,QDIST,symmetric_hard_clipping,low,2,2,1.861311
6,QGAIN,smooth_time_varying_gain,high,2,2,0.912562
7,QGAIN,smooth_time_varying_gain,low,2,2,0.863203
8,QGAIN,uniform_level_shift,high,2,2,0.776335
9,QGAIN,uniform_level_shift,low,2,2,1.497990


PAIRED ΔQ / ΔA TABLE: WRITTEN
No clinical outcome, prediction, loss, or model object was loaded.



## 10. Preflight seal

A PASS here means only that the controlled **measurement engine** is ready.  
Clinical Goal 3 inference remains blocked until the authoritative Goal 2 final seal and model-bundle reproduction gate pass.


In [18]:

output_files = [
    path
    for path in OUT.rglob("*")
    if path.is_file()
    and path.name not in {"PRE_FLIGHT_SUCCESS.json"}
]

output_hashes = {
    str(path.relative_to(OUT)): sha256_file(path)
    for path in sorted(output_files)
}

preflight_manifest = {
    "created_utc": utc_now(),
    "status": "PASS",
    "engine_version": ENGINE_VERSION,
    "paper1_commit": PAPER1_COMMIT,
    "acoustic_engine": ACOUSTIC_ENGINE,
    "a_freeze_version": A_FREEZE_VERSION,
    "stage_b_manifest_sha256": manifest_hash,
    "a_registry_sha256": a_freeze_manifest["a_registry_sha256"],
    "a_values_sha256": a_freeze_manifest["acoustic_features_frozen_sha256"],
    "preflight_sources": int(EXPECTED["preflight_sources"]),
    "sealed_transform_count": int(dose_manifest["transform"].nunique()),
    "smoke_execution_rows": int(len(response)),
    "baseline_a_reproduction_comparisons": int(len(reproduction)),
    "baseline_a_reproduction_failures": int(len(failed_reproduction)),
    "fresh_resegmentation_exact_match_required": False,
    "primary_goal3_segmentation_rule": "rerun_segmentation",
    "fixed_frozen_segmentation_rule": "prespecified_sensitivity",
    "clinical_outcomes_loaded": False,
    "goal2_predictions_loaded": False,
    "goal2_model_objects_loaded": False,
    "clinical_losses_loaded": False,
    "full_controlled_clinical_experiment_run": False,
    "next_allowed_step": (
        "After authoritative Goal2 FINAL seal: construct five fold-specific "
        "Goal2 model bundles, reproduce repeat-1 OOF predictions, seal bundles, "
        "then execute the full controlled Goal3 experiment."
    ),
    "output_hashes": output_hashes,
}

atomic_json(
    preflight_manifest,
    OUT / "PRE_FLIGHT_SUCCESS.json",
)

print("=" * 76)
print("GOAL 3 CONTROLLED Q+A MEASUREMENT PREFLIGHT: PASS")
print("=" * 76)
print("Frozen-A implementation reproduction using frozen segmentation: PASS")
print("Six sealed perturbation structures exercised: PASS")
print("Q + A extraction after re-segmentation: PASS")
print("Clinical outcomes loaded: FALSE")
print("Goal 2 predictions/models loaded: FALSE")
print("Full clinical perturbation experiment run: FALSE")
print()
print("WAITING DEPENDENCY:")
print("Goal 2 FINAL seal + exact model-bundle OOF reproduction.")


GOAL 3 CONTROLLED Q+A MEASUREMENT PREFLIGHT: PASS
Frozen-A implementation reproduction using frozen segmentation: PASS
Six sealed perturbation structures exercised: PASS
Q + A extraction after re-segmentation: PASS
Clinical outcomes loaded: FALSE
Goal 2 predictions/models loaded: FALSE
Full clinical perturbation experiment run: FALSE

WAITING DEPENDENCY:
Goal 2 FINAL seal + exact model-bundle OOF reproduction.
